In [1]:
import requests
import urllib.parse
from bs4 import BeautifulSoup
import mpl_finance
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from datetime import datetime
from openpyxl import load_workbook
import plotly.graph_objects as go
from plotly import subplots

%run _CrawBase.ipynb
%run _BaseInfo.ipynb

df_info=_baseInfo.copy()

C:\Users\user\anaconda3\lib\site-packages\mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


[上市] 線上抓取成功，共 1090 筆
[上櫃] 線上抓取成功，共 892 筆
[output] 儲存成功，共 1982 筆
df_info 共 1980 筆，上市：1089，上櫃：891


In [2]:
def roc_to_gregorian(roc_date_str):
    # 解析字串，假設格式為 "ROC_YEAR/MM/DD"
    roc_year, month, day = roc_date_str.split('/')
    # 轉換為整數
    roc_year = int(roc_year)
    month = int(month)
    day = int(day)
    
    # ROC 年轉西元年
    gregorian_year = roc_year + 1911
    
    # 返回格式化的字串，或是你可以選擇返回 datetime.date 物件
    return pd.to_datetime(f"{gregorian_year:04d}-{month:02d}-{day:02d}")

In [3]:
def save_data_by_date(df, output_dir):
    # 清理无效字符（如 '*'）
    df['日期'] = df['日期'].astype(str).str.replace('*', '', regex=False)

    # 转换为日期格式
    df['日期'] = pd.to_datetime(df['日期'], format='%Y/%m/%d', errors='coerce')

    # 创建存储结果的文件夹
    os.makedirs(output_dir, exist_ok=True)

    # 按日期分组并保存为独立的 Excel 文件
    for date, group in df.groupby('日期'):
        print(date)
        print(roc_to_gregorian(min(end_time_list)))
        if(date>=roc_to_gregorian(min(end_time_list))):
            formatted_date = date.strftime('%Y-%m-%d')  # 格式化日期为文件名
            file_name = f"{output_dir}/{formatted_date}.xlsx"

            # 保存每个分组为单独的 Excel 文件
            group.to_excel(file_name, index=False)

In [4]:
# 計算 RSI 指標
def calculate_rsi(data, window=14):
    delta = data['收盤價'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [5]:
def _prepare_interval_stats(data_array):
    """預先計算信賴區間所需統計量，避免在 apply 中重複運算。"""
    cleaned = [float(v) for v in data_array if _is_valid_positive(v)]
    if len(cleaned) < 2:
        return cleaned, 0.0, 1.0
    mean    = np.mean(cleaned)
    std_dev = np.std(cleaned, ddof=1)
    return cleaned, mean, std_dev

def _is_valid_positive(v):
    try:
        return float(v) > 0
    except (ValueError, TypeError):
        return False

def get_interval(value_to_check, data, _mean=None, _std_dev=None):
    """
    計算 value_to_check 所在的最小信賴區間。
    _mean / _std_dev 可從外部傳入（避免在 apply 中重複計算）。
    """
    if _mean is None or _std_dev is None:
        _, _mean, _std_dev = _prepare_interval_stats(data)

    value_to_check = float(value_to_check)
    confidence_levels = [0.1, 0.30, 0.50, 0.65, 0.90, 0.99]

    for level in confidence_levels:
        z_score         = stats.norm.ppf(1 - (1 - level) / 2)
        margin_of_error = z_score * _std_dev
        lower_bound     = _mean - margin_of_error
        upper_bound     = _mean + margin_of_error
        if lower_bound <= value_to_check <= upper_bound:
            interval_type = "正區間" if _mean < value_to_check else "負區間"
            return (level, interval_type, lower_bound, upper_bound)

    interval_type = "正區間" if _mean < value_to_check else "負區間"
    return (1, interval_type, value_to_check, value_to_check)


In [6]:
# 清理函式，只對字串型態欄位進行處理
def clean_column(column):
    if column.dtype == 'object':
        return column.str.replace(',', '').replace('0', np.nan).replace('--', '').apply(pd.to_numeric, errors='coerce')
    return column  # 如果已經是數值型態，直接回傳

In [7]:
#!!!!!!!!

In [8]:
def find_pivots(df, n=3):
    """向量化找 pivot high / low，比雙重迴圈快 10-50 倍。"""
    high = df['最高價']
    low  = df['最低價']

    # rolling max/min（視窗 2n+1，center=True）
    roll_max = high.rolling(2 * n + 1, center=True, min_periods=2 * n + 1).max()
    roll_min = low.rolling(2 * n + 1, center=True, min_periods=2 * n + 1).min()

    pivot_high_mask = high == roll_max
    pivot_low_mask  = low  == roll_min

    highs = list(zip(df.index[pivot_high_mask], high[pivot_high_mask]))
    lows  = list(zip(df.index[pivot_low_mask],  low[pivot_low_mask]))

    # 轉成 (positional_index, value) 格式保持向下相容
    idx_map = {idx: pos for pos, idx in enumerate(df.index)}
    highs = [(idx_map[i], v) for i, v in highs]
    lows  = [(idx_map[i], v) for i, v in lows]

    return highs, lows


def calc_trend_features(df):
    """向量化計算 HH/HL/MA_UP/VOL_UP/TrendScore，避免逐列迴圈。"""
    highs, lows = find_pivots(df)

    n = len(df)

    # ── 把 pivot 資訊轉成「截至當天的最後 3 個值」序列 ──
    # 用 numpy 逐 pivot 更新累積陣列，時間複雜度 O(pivot 數 + n)
    high_vals = np.full((n, 3), np.nan)  # [h1, h2, h3] 最舊→最新
    low_vals  = np.full((n, 3), np.nan)

    _hq = []  # 當前累積的 pivot 高點佇列
    _lq = []

    hi_ptr = 0
    lo_ptr = 0

    for i in range(n):
        while hi_ptr < len(highs) and highs[hi_ptr][0] <= i:
            _hq.append(highs[hi_ptr][1])
            hi_ptr += 1
        while lo_ptr < len(lows) and lows[lo_ptr][0] <= i:
            _lq.append(lows[lo_ptr][1])
            lo_ptr += 1
        if len(_hq) >= 3:
            high_vals[i] = _hq[-3], _hq[-2], _hq[-1]
        if len(_lq) >= 3:
            low_vals[i]  = _lq[-3], _lq[-2], _lq[-1]

    # HH：最近三個高點遞增
    HH = (high_vals[:, 2] > high_vals[:, 1]) & (high_vals[:, 1] > high_vals[:, 0])
    HH = np.where(np.isnan(high_vals[:, 0]), False, HH)

    # HL：最近三個低點遞增
    HL = (low_vals[:, 2] > low_vals[:, 1]) & (low_vals[:, 1] > low_vals[:, 0])
    HL = np.where(np.isnan(low_vals[:, 0]), False, HL)

    # MA_UP / VOL_UP（向量化）
    ma5_up  = df['MA_5'].values  > np.roll(df['MA_5'].values,  1)
    ma20_up = df['MA_20'].values > np.roll(df['MA_20'].values, 1)
    ma5_gt_ma20 = df['MA_5'].values > df['MA_20'].values
    MA_UP = ma5_up & ma20_up & ma5_gt_ma20
    MA_UP[0] = False  # 第一列 shift 無效

    VOL_UP = df['Volume_MA_5'].values > df['Volume_MA_50'].values

    score = (HH.astype(int) * 40
             + HL.astype(int) * 30
             + MA_UP.astype(int) * 20
             + VOL_UP.astype(int) * 10)

    df['HH']         = HH
    df['HL']         = HL
    df['MA_UP']      = MA_UP
    df['VOL_UP']     = VOL_UP
    df['TrendScore'] = score

    df['TrendType'] = np.select(
        [df['TrendScore'] >= 80, df['TrendScore'] >= 60,
         df['TrendScore'] >= 40, df['TrendScore'] >= 20],
        ['強多', '轉強', '盤整偏多', '弱勢整理'],
        default='空頭'
    )

    df['TrendScorertxt'] = (
        'TrendType : '  + df['TrendType'].astype(str)  + '<br>' +
        'TrendScore : ' + df['TrendScore'].astype(str) + '<br>' +
        'HH : '         + df['HH'].astype(str)         + '<br>' +
        'HL : '         + df['HL'].astype(str)         + '<br>' +
        'MA : '         + df['MA_UP'].astype(str)      + '<br>' +
        'VOL : '        + df['VOL_UP'].astype(str)
    )

    return df


### Main 


In [9]:
def data_process(RowData_df_craw_stock):
    # 基本資料清理
    df = RowData_df_craw_stock.copy().drop_duplicates()
    df['日期'] = df['日期'].str.replace("＊", "", regex=False)
    df['日期'] = df['日期'].str.extract(r'(\d{2,3})/(\d{1,2}/\d{1,2})').apply(
        lambda x: f"{int(x[0]) + 1911}/{x[1]}", axis=1
    )
    df['年月日'] = df['日期']

    # 數值欄位轉換
    cols_to_clean = ['成交金額', '收盤價', '開盤價', '最低價', '最高價', '成交股數', '漲跌價差', '成交筆數']
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = clean_column(df[col])

    df['收盤價'] = pd.to_numeric(df['收盤價'], errors='coerce').fillna(method='ffill')
    
    # 前日收盤
    df['前日收盤價'] = df['收盤價'].shift(1)
    df['最小收盤價']= df['收盤價'].min()
    
    # ===== 技術指標計算區 =====
    ###########       【價】      ######################
    # MACD 計算　 MACD（指數平滑異同移動平均線）
    ema_12 = df['收盤價'].ewm(span=12, adjust=False).mean()
    ema_26 = df['收盤價'].ewm(span=26, adjust=False).mean()
    df['MACD'] = ema_12 - ema_26
    df['MACD-SL'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD-SL_max']= df['MACD-SL'].max()
    df['MACD-SL_min']= df['MACD-SL'].min()
   
    # ★ MACD 柱狀圖：快線 - 慢線
    df['MACD_hist'] = df['MACD'] - df['MACD-SL']
    df['MACD_hist_diff']=df['MACD_hist']-df['MACD_hist'].shift(1)

    # ========= MACD 口訣訊號 =========

    # 金叉／死叉（當天）
    macd_golden = (df['MACD'] > df['MACD-SL']) & (df['MACD'].shift(1) <= df['MACD-SL'].shift(1))
    macd_death  = (df['MACD'] < df['MACD-SL']) & (df['MACD'].shift(1) >= df['MACD-SL'].shift(1))

    # 允許「近 K 天內有金叉 / 死叉」就算（而不是一定當天）
    K = 15
    macd_golden_recent = macd_golden.rolling(K, min_periods=1).max().astype(bool)
    macd_death_recent  = macd_death.rolling(K, min_periods=1).max().astype(bool)

    # =========== 抄底：前大後小 + 金叉 ===========

    # 柱狀在 0 軸下方，今天比昨天「沒那麼負」= 空方減弱
    hist_neg = df['MACD_hist'] < 0
    hist_contract = hist_neg & (df['MACD_hist'] > df['MACD_hist'].shift(1))

    df['MACD_bottom_signal'] = hist_contract & macd_golden_recent

    # =========== 逃頂：前高後低 + 死叉 + 放量 ===========

    # 柱狀在 0 軸上方，今天比昨天「更小」= 多頭動能衰退
    hist_pos = df['MACD_hist'] > 0
    hist_shrink = hist_pos & (df['MACD_hist'] < df['MACD_hist'].shift(1))

    # 放量：用近 10 日均量，之後你覺得太鬆再放大倍率
    vol_ma10 = df['成交金額'].rolling(window=10, min_periods=1).mean()
    vol_boost = df['成交金額'] > vol_ma10

    # 也允許「近幾天內有放量」，不要綁死當天
    vol_boost_recent = vol_boost.rolling(K, min_periods=1).max().astype(bool)

    df['MACD_top_signal'] = hist_shrink & macd_death_recent & vol_boost_recent


    # ========= 起漲區：MACD 站上 0 軸後一路走高 =========
    # 在 0 軸上方，且柱狀比前一天大 → 多頭動能在增強
    hist_pos = df['MACD_hist'] > 0
    hist_up  = df['MACD_hist'] > df['MACD_hist'].shift(1)

    # 整段「起漲區」：MACD 在 0 軸上方且動能在放大
    df['MACD_up_run'] = hist_pos & hist_up

    # 起漲「起點」：這段 run 的第一天
    up_run = df['MACD_up_run']
    df['MACD_rally_start'] = up_run & ~up_run.shift(1).fillna(False)
    
    # ========= =========
    
    # MACD 黃金交叉判斷
    df['MACD_golden_cross'] = ((df['MACD'] > df['MACD-SL']) & (df['MACD'].shift(1) <= df['MACD-SL'].shift(1)))
    df['MACD_last_cross_date'] = df.loc[df['MACD_golden_cross'], '年月日'].max()

    try:
        MACD_last_cross_date=df.loc[df['MACD_golden_cross'], '年月日'].max()
        df['MACD_last_cross_date'] = MACD_last_cross_date
        df['MACD_last_cross_date收盤價']=df[df['年月日']==MACD_last_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print(f'Error get MACD_last_cross_date : {error}')   
    
    
    df['MACD_death_cross'] = ((df['MACD'] < df['MACD-SL']) & (df['MACD'].shift(1) >= df['MACD-SL'].shift(1)))
    MACD_death_cross_date=df.loc[df['MACD_death_cross'], '年月日'].max()
    
    try:
        MACD_death_cross_date=df.loc[df['MACD_death_cross'], '年月日'].max()
        df['MACD_death_cross_d'] = MACD_death_cross_date
        df['MACD_death_cross_d收盤價']=df[df['年月日']==MACD_death_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print(f'Error get MACD_death_cross : {error}') 
        
        
    try:
        MACD_death_cross_date2=df.loc[df['MACD_death_cross'], '年月日'].iloc[-2]
        df['MACD_death_cross_d2'] = MACD_death_cross_date2
        df['MACD_death_cross_d2收盤價']=df[df['年月日']==MACD_death_cross_date2]['收盤價'].iloc[0]
    except Exception as error:
        print(f'Error get MACD_death_cross_date2 : {error}')    

    # KD 計算
    low9 = df['收盤價'].rolling(window=9).min()
    high9 = df['收盤價'].rolling(window=9).max()
    df['%K'] = (df['收盤價'] - low9) / (high9 - low9) * 100
    df['%D'] = df['%K'].rolling(window=3).mean()

    # KD 黃金交叉判斷
    df['KD_golden_cross'] = ((df['%K'] > df['%D']) & (df['%K'].shift(1) <= df['%D'].shift(1)))
    
    # MA 計算　MA（移動平均線）與交叉
    df['MA_5'] = df['收盤價'].rolling(window=5).mean()
    df['MA_10'] = df['收盤價'].rolling(window=10).mean()
    df['MA_20'] = df['收盤價'].rolling(window=20).mean()
    df['MA_50'] = df['收盤價'].rolling(window=50).mean()
    df['MA_80'] = df['收盤價'].rolling(window=80).mean()
    df['MA_240'] = df['收盤價'].rolling(window=240).mean()
    
    df['MA_5_斜率'] = df['MA_5'].pct_change()
    df['MA_10_斜率'] = df['MA_10'].pct_change()
    df['MA_20_斜率'] = df['MA_20'].pct_change()
    df['MA_50_斜率'] = df['MA_50'].pct_change()
    df['MA_80_斜率'] = df['MA_80'].pct_change()
    df['MA_240_斜率'] = df['MA_240'].pct_change()

    
    df['MA_break'] = ((df['MA_5'] > df['MA_20'])  & 
                     (df['MA_5'].shift(3) <= df['MA_20'].shift(3))&
                     (df['MA_5'] > df['MA_80']) )

    # MA 20 vs 50　黃金交叉判斷
    df['MA_golden_cross'] = ((df['MA_20'] > df['MA_50']) & (df['MA_20'].shift(1) <= df['MA_50'].shift(1)))

    try:
        MA_last_cross_date=df.loc[df['MA_golden_cross'], '年月日'].max()
        df['MA_last_cross_date'] = MA_last_cross_date
        df['MA_last_cross_date收盤價']=df[df['年月日']==MA_last_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print("")
        #print(f'Error get MA_last_cross_d2 : {error}')   
        
    # MA 5 vs  線價　黃金交叉判斷
    df['MA_5_golden_cross'] = ((df['收盤價'] > df['MA_5']) & (df['收盤價'].shift(1) <= df['MA_5'].shift(1)))
    try:
        MA_5_last_cross_date=df.loc[df['MA_5_golden_cross'], '年月日'].max()
        df['MA_5_last_cross_date'] = MA_5_last_cross_date
        df['MA_5_last_cross_date收盤價']=df[df['年月日']==MA_5_last_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print("")
    
            
    # MA 死亡交叉判斷
    df['MA_death_cross'] = ((df['MA_20'] < df['MA_50']) & (df['MA_20'].shift(1) >= df['MA_50'].shift(1)))
    try:
        MA_death_cross_date=df.loc[df['MA_death_cross'], '年月日'].max()
        df['MA_death_cross_d'] = MA_death_cross_date
        df['MA_death_cross_d收盤價']=df[df['年月日']==MA_death_cross_date]['收盤價'].iloc[0]
    except Exception as error:
        print("")
        #print(f'Error get MA_death_cross_date : {error}') 
        
    try:
        MA_death_cross_date2=df.loc[df['MA_death_cross'], '年月日'].iloc[-2]
        df['MA_death_cross_d2'] = MA_death_cross_date2
        df['MA_death_cross_d2收盤價']=df[df['年月日']==MA_death_cross_date2]['收盤價'].iloc[0]
    except Exception as error:
        print("")
        #print(f'Error get MA_death_cross_date2 : {error}')    
    
    
    # RSI
    df['RSI'] = calculate_rsi(df)
    df['RSI_rebound'] = (df['RSI'] > 30) & (df['RSI'].shift(1) <= 30)
    ############################################
    # ===== 近 N 天低 / 高點與發生日（numpy 向量化，避免 rolling apply lambda）=====
    _shifted_prices = df['收盤價'].shift(1).values
    _dates_arr      = df['年月日'].values
    _n_rows         = len(df)

    for _n in [15, 30, 60, 90]:
        _low_vals  = np.full(_n_rows, np.nan)
        _high_vals = np.full(_n_rows, np.nan)
        _low_dates  = np.empty(_n_rows, dtype=object)
        _high_dates = np.empty(_n_rows, dtype=object)

        for _i in range(_n_rows):
            _start = max(0, _i - _n + 1)
            _win   = _shifted_prices[_start:_i + 1]
            _valid = _win[~np.isnan(_win)]
            if len(_valid) == 0:
                continue
            _win_dates = _dates_arr[_start:_i + 1]
            _nan_mask  = ~np.isnan(_win)

            _low_pos  = np.nanargmin(_win)
            _high_pos = np.nanargmax(_win)

            _low_vals[_i]   = _win[_low_pos]
            _high_vals[_i]  = _win[_high_pos]
            _low_dates[_i]  = _win_dates[_low_pos]
            _high_dates[_i] = _win_dates[_high_pos]

        df[f'Low_{_n}d']       = _low_vals
        df[f'Low_{_n}d_date']  = _low_dates
        df[f'High_{_n}d']      = _high_vals
        df[f'High_{_n}d_date'] = _high_dates
    ############################################
    
    ###########       【量】      ######################
    
   # 價格變動方向加權成交金額（）１
    
    df['成交金額_5MA'] = df['成交金額'].rolling(window=5, min_periods=1).mean()

    # 如果成交金額是 NaN → 使用 5MA 補值
    # //Volume_Price_Change
    df['成交金額_補值']= df['成交金額'].fillna(df['成交金額_5MA'])

    # 成交量移動平均與震盪指標
    df['Volume_MA_5'] = df['成交金額_補值'].rolling(window=5).mean()
    df['Volume_MA_10'] = df['成交金額_補值'].rolling(window=10).mean()
    df['Volume_MA_30'] = df['成交金額_補值'].rolling(window=30).mean()
    df['Volume_MA_50'] = df['成交金額_補值'].rolling(window=50).mean()
    df['Volume_MA_80'] = df['成交金額_補值'].rolling(window=80).mean()
    
    df['量短5長30比'] = (df['Volume_MA_5'] / df['Volume_MA_30']) 
    df['量短5長30比_avg'] = df['量短5長30比'].mean()
    df['量短5長30比'] = np.where((df['量短5長30比'] < df['量短5長30比_avg']) |
                             (df['Volume_MA_10'] < df['Volume_MA_50']) 
                             ,0
                             ,df['量短5長30比'])

    # VPC MACD 系列
    df['VPC_MACD'] = df['成交金額_補值'].ewm(span=9, adjust=False).mean() - \
                     df['成交金額_補值'].ewm(span=15, adjust=False).mean()
    df['VPC_SIGNAL'] = df['VPC_MACD'].ewm(span=9, adjust=False).mean()
    df['VPC_DIF'] = df['VPC_MACD'] - df['VPC_SIGNAL']
        
    #成交金額 均　
    ###### 計算最大與最小日期的相差天數
    df['年月日'] = pd.to_datetime(df['年月日'].astype(str).str.replace('*', '', regex=False), format='%Y-%m-%d')

    # 取最大與最小日期
    max_date = df['年月日'].max()
    min_date = df['年月日'].min()

    # 計算天數差
    diff_days = (max_date - min_date).days

    
    total_amount = df['成交金額'].sum()
    avg_amount = df['成交金額'].mean()
    
    df['成交金額平均'] = avg_amount*1.5
    
    
    df['voc_cross'] = ((df['Volume_MA_10'] > df['Volume_MA_50']) 
                       & (df['Volume_MA_10'].shift(1) <= df['Volume_MA_50'].shift(1))
                      )
    voc_cross_date=df.loc[df['voc_cross'], '年月日'].max()
    
    try:
        voc_cross_date=df.loc[df['voc_cross'], '年月日'].max()
        df['voc_cross_d'] = voc_cross_date
        
    except Exception as error:
        print("")
        #print(f'Error get voc_cross_date : {error}') 
        
    
    
    ######

    # 交叉與門檻
   # 動能交叉幅度門檻：改小一些，讓條件不太嚴苛
    threshold = df['VPC_DIF'].rolling(window=10).std() * 0.1
    # volume_threshold 改用穩定的長期平均，避免波動太大
    df['volume_threshold'] = df['VPC_MACD'].ewm(span=12, adjust=False).mean()
    # 成交金額門檻：用絕對成交金額平均（非加權）來代表市場活躍程度
    volume_threshold = df['成交金額'].rolling(window=10).mean()
    
    # 黃金交叉觸發條件
    df['成交金額_補值_break'] = (
        (df['VPC_MACD'] > df['VPC_SIGNAL']) &
        (df['VPC_MACD'] > df['volume_threshold']) &
        (df['VPC_DIF'] > threshold) & 
        (df['成交金額'] > volume_threshold) & # 活躍才觸發
        (df['VPC_MACD'].shift(1) >= df['VPC_SIGNAL'].shift(1)) # 過去交叉持續
    )
    
    # 成交量擴增與變動率
    
    # 資料時間轉換與亮點處理
    df['年月日'] = pd.to_datetime(df['年月日'].astype(str).str.replace('*', '', regex=False), format='%Y-%m-%d')
    df, merged_intervals = get_highlight(df)
    ###########################################################################################
    
    # MACD 上升趨勢
    df['macd_golden_crosses_area'] = (
        (df['MACD'] > df['MACD-SL']) &
        (df['MACD'].shift(1) >= df['MACD-SL'].shift(1))
    )

    # =====  UI 顯示顏色標記=====
    df['Bar_Color'] = np.where(df['收盤價'].diff() > 0, 'red', 'green')
    
    # ===== 分類邏輯區 =====
    df['Full_Summary'] = ''
    df = full_technical_analysis(df)

    
    #============================================================================================================
    
    df = add_bollinger_bands(df)
    
    ###########################################################################################
    # 均線剛翻多（今天才第一次 MA_5 > MA_20 > MA_50）
    cond_ma_now = (df['MA_5'] > df['MA_20']) & (df['MA_20'] > df['MA_50'])
    ma_flip_today = cond_ma_now & ~(cond_ma_now.shift(1).fillna(False))

    # 量能放大（成交金額大於短期均量的 1.1 倍；原本 1.2 → 放寬一點）
    vol_expand_today = df['成交金額'] > (df['Volume_MA_5'] * 1.1)

    # MACD：改為「在零軸上方」即可（原本只抓『剛突破零軸』會漏掉後半段）
    macd_positive = df['MACD'] > 0
    # 若你仍想保留「剛破零軸」的嚴格條件，也一起算出來（供 rolling 使用）
    macd_zero_today = (df['MACD'] > 0) & (df['MACD'].shift(1) <= 0)

    # 允許近 K 天內先後滿足（K 可調）
    K = 3
    # 均線：用「正在多頭排列」以涵蓋後半段；若要嚴格起點，改回 ma_flip_today
    #ma_recent   = cond_ma_now.rolling(K, min_periods=1).max().astype(bool)
    ma_recent   = ma_flip_today.rolling(K, min_periods=1).max().astype(bool)
    vol_recent  = vol_expand_today.rolling(K, min_periods=1).max().astype(bool)
    # MACD：允許近 K 日內曾經『剛破零軸』或當下已 > 0
    macd_recent = ((macd_zero_today | macd_positive)
                   .rolling(K, min_periods=1).max().astype(bool))

    # 綜合條件（主升段起點/進場點）
    cond_entry = ma_recent & vol_recent & macd_recent
    df['cond_entry'] =( ma_recent & vol_recent & macd_recent )
    
    ###########################################################################################
    # 百分比欄位：直接乘 100 round 到小數兩位，跳過 apply format 再 astype 的繞路
    df['MA_5_%']      = ((df['收盤價'] - df['MA_5'])  / df['MA_5'].abs()  * 100).round(2)
    df['均價_%']       = ((df['MA_5']  - df['MA_20']) / df['MA_20'].abs() * 100).round(2)
    df['均價long_%']   = ((df['MA_20'] - df['MA_50']) / df['MA_50'].abs() * 100).round(2)
    df['MACD_%']      = ((df['MACD']  - df['MACD-SL']) / df['MACD-SL'].abs() * 100).round(2)

    
    ###########################################################################################
    #處理 條件分區 
    ###########################################################################################
    
    cond_a = (
        (df['MA_5_%'] > 0) &
        (df['均價_%'] > 0) &
        (df['MACD'] > df['MACD-SL']) &
        (df['Volume_MA_5'] >df['Volume_MA_50']*0.8 ) 
    )
    cond_b1 = (df['均價long_%'] > 0) & (df['MACD_%'] > 0) & (df['均價_%'] > 0)  
    cond_b2 = (df['均價long_%'] > 0) & (df['MACD_%'] > 0) & (df['均價_%'] < 0)  
    cond_c =  (df['均價long_%'] > 0) & (df['MACD_%'] < 0)
    cond_d1 = (df['均價long_%'] < 0) & (df['MACD_%'] > 0) & (df['均價_%'] > 0)  
    cond_d2 = (df['均價long_%'] < 0) & (df['MACD_%'] > 0) & (df['均價_%'] < 0)  
    cond_e =  (df['均價long_%'] < 0) & (df['MACD_%'] < 0)

    def add_tag(df, condition, label):
        # 僅在分類標籤中尚未包含該類別時才加入
        df.loc[condition & (~df['分類'].str.contains(label, na=False)), '分類'] += f'{label};'

    '''
    # 使用 np.select 依條件分類
    conditions = [cond_a,cond_b1,cond_b2, cond_c, cond_d1, cond_d2, cond_e]
    choices =    ['價_量', 'B1分流','B2分流', 'C分流', 'D1分流','D2分流', 'E分流']

    df['分類'] = np.select(conditions, choices, default='未分類')
    ''' 
    df['分類'] =''
    add_tag(df, cond_a, '價_量')
    add_tag(df, cond_b1, 'B1分流')
    add_tag(df, cond_b2, 'B2分流')
    add_tag(df, cond_c,  'C分流')
    add_tag(df, cond_d1, 'D1分流')
    add_tag(df, cond_d2, 'D2分流')
    add_tag(df, cond_e,  'E分流')

    ###########################################################################################
    #處理數據信賴區間
    ###########################################################################################
    
    
    # 信賴區間：提前算好 mean/std，避免每列重複計算
    data_array = np.array(df['收盤價'])
    _ci_cleaned, _ci_mean, _ci_std = _prepare_interval_stats(data_array)

    results = [
        get_interval(v, _ci_cleaned, _mean=_ci_mean, _std_dev=_ci_std)
        for v in df['收盤價']
    ]

    df[['level', 'interval_type', 'lower_bound', 'upper_bound']] = pd.DataFrame(
        results, index=df.index
    )
    ###########################################################################################
    df=calc_trend_features(df)

    ###########################################################################################
    # StateScore：強勢 / 盤整 / 弱勢 三態判斷
    # 分三個維度加權：價結構 50分 ＋ 量確認 20分 ＋ MACD 30分
    ###########################################################################################

    # ── 【價】50分 ──
    # 25分：收盤價站在 MA20 之上
    price_above_ma20 = (df['收盤價'] > df['MA_20']).astype(int) * 25
    # 15分：MA20 斜率向上（今天比昨天高）
    ma20_slope_up = (df['MA_20'] > df['MA_20'].shift(1)).astype(int) * 15
    # 10分：MA20 在 MA60 之上（中期結構健康）
    ma20_above_ma60 = (df['MA_20'] > df['MA_50']).astype(int) * 10

    # ── 【量】20分 ──
    # 20分：短期量（5日均）> 長期量（30日均），代表量能活躍
    vol_active = (df['Volume_MA_5'] > df['Volume_MA_30']).astype(int) * 20

    # ── 【MACD】30分 ──
    # 20分：MACD 站在零軸上方
    macd_positive_score = (df['MACD'] > 0).astype(int) * 20
    # 10分：MACD 柱狀圖在擴大（動能增強）
    macd_expanding = (df['MACD_hist'] > df['MACD_hist'].shift(1)).astype(int) * 10

    # ── 合計（滿分 100 分）──
    df['StateScore'] = (
        price_above_ma20 +
        ma20_slope_up +
        ma20_above_ma60 +
        vol_active +
        macd_positive_score +
        macd_expanding
    ).clip(0, 100)

    # ── 三態標籤（強勢≥90 / 盤整≥20 / 弱勢<20）──
    df['MarketState'] = np.select(
        [df['StateScore'] >= 90, df['StateScore'] >= 20],
        ['強勢', '盤整'],
        default='弱勢'
    )
    
    df['MarketState_prev'] = df['MarketState'].shift(1)

    # ── hover 用文字 ──
    # 閾值：強勢≥90 / 盤整≥20
    df['StateScore_txt'] = (
        'StateScore : ' + df['StateScore'].astype(str) + '<br>' +
        'MarketState : ' + df['MarketState'] + '<br>' +
        '價在MA20上 : ' + price_above_ma20.astype(str) + '/25<br>' +
        'MA20斜率向上 : ' + ma20_slope_up.astype(str) + '/15<br>' +
        'MA20>MA60 : ' + ma20_above_ma60.astype(str) + '/10<br>' +
        '量能活躍 : ' + vol_active.astype(str) + '/20<br>' +
        'MACD>0 : ' + macd_positive_score.astype(str) + '/20<br>' +
        'MACD擴張 : ' + macd_expanding.astype(str) + '/10'
    )


    ###########################################################################################
    # 入場點 / 出場點計算
    # 邏輯：趨勢過濾 + 動能確認 + 低檔觸發
    # 入場：MA多頭排列 + MACD站上零軸且擴張 + KD低檔金叉（或收盤剛站回MA20）
    # 出場：MACD柱連3日縮小 / KD超買反轉 / 收盤跌破MA20
    ###########################################################################################

    # ── 【入場】條件 1：趨勢確認（中期多頭，過濾下跌股）──
    trend_ok = (
        (df['MA_20'] > df['MA_50']) &                          # MA20 在 MA60 上方
        (df['MA_20'] > df['MA_20'].shift(1))                   # MA20 斜率向上
    )

    # ── 【入場】條件 2：動能確認（MACD 站上零軸且擴張）──
    momentum_ok = (
        (df['MACD'] > 0) &                                     # MACD 在零軸上方
        (df['MACD_hist'] > df['MACD_hist'].shift(1))           # 柱狀圖擴大
    )

    # ── 【入場】條件 3：觸發訊號（低檔進場，不追高）──
    trigger_entry = (
        # 觸發 A：KD 低檔金叉（%K < 50 避免追高）
        (df['KD_golden_cross'] & (df['%K'] < 50)) |
        # 觸發 B：收盤剛站回 MA20 上方（前一日在下方）
        (
            (df['收盤價'] > df['MA_20']) &
            (df['收盤價'].shift(1) <= df['MA_20'].shift(1))
        )
    )

    # ── 合併入場訊號 ──
    df['Entry_Signal'] = trend_ok & momentum_ok & trigger_entry

    # ── 【出場】條件（任一觸發）──

    # 條件 A：MACD 柱狀連續 3 日縮小（動能衰退，早出）
    hist_shrink = df['MACD_hist'] < df['MACD_hist'].shift(1)
    exit_momentum = (
        hist_shrink &
        hist_shrink.shift(1).fillna(False) &
        hist_shrink.shift(2).fillna(False)
    )

    # 條件 B：KD 超買反轉（%K > 80 且今天低於昨天）
    exit_kd_overbought = (
        (df['%K'] > 80) &
        (df['%K'] < df['%K'].shift(1))
    )

    # 條件 C：收盤跌破 MA20
    exit_break_ma20 = (
        (df['收盤價'] < df['MA_20']) &
        (df['收盤價'].shift(1) >= df['MA_20'].shift(1))
    )

    # ── 合併出場訊號 ──
    df['Exit_Signal'] = exit_momentum | exit_kd_overbought | exit_break_ma20

    # ── StopLoss：只在入場後 20 天內監控，跌幅 -7% 才標記 ──
    # 原則：沒有入場就沒有停損，避免空頭股滿圖橘叉
    STOP_LOSS_PCT = 0.07
    STOP_WINDOW   = 20        # 入場後幾天內監控停損

    # StopLoss 向量化：對每個入場點，找 window 內第一個跌幅 >= STOP_LOSS_PCT 的位置
    prices      = df['收盤價'].values
    entry_mask  = df['Entry_Signal'].values
    stop_arr    = np.zeros(len(df), dtype=bool)

    entry_positions = np.where(entry_mask)[0]
    for ep in entry_positions:
        ep_price = prices[ep]
        end      = min(ep + STOP_WINDOW + 1, len(df))
        hits     = np.where(
            (prices[ep+1:end] - ep_price) / ep_price <= -STOP_LOSS_PCT
        )[0]
        if len(hits):
            stop_arr[ep + 1 + hits[0]] = True

    df['StopLoss_Signal'] = stop_arr

    # ── Hover 說明文字 ──
    df['Entry_txt'] = ''
    df.loc[df['Entry_Signal'], 'Entry_txt'] = (
        '【入場訊號】<br>' +
        'MA20>MA60: ✅<br>' +
        'MACD>0 且擴張: ✅<br>' +
        'KD%K: ' + df.loc[df['Entry_Signal'], '%K'].round(1).astype(str) + '<br>' +
        'StateScore: ' + df.loc[df['Entry_Signal'], 'StateScore'].astype(str) + '<br>' +
        '收盤價: ' + df.loc[df['Entry_Signal'], '收盤價'].astype(str)
    )
    df['Exit_txt'] = ''
    df.loc[df['Exit_Signal'], 'Exit_txt'] = (
        '【出場訊號】<br>' +
        'MACD柱縮3日: ' + exit_momentum.loc[df['Exit_Signal']].astype(str) + '<br>' +
        'KD超買反轉: ' + exit_kd_overbought.loc[df['Exit_Signal']].astype(str) + '<br>' +
        '跌破MA20: ' + exit_break_ma20.loc[df['Exit_Signal']].astype(str) + '<br>' +
        '收盤價: ' + df.loc[df['Exit_Signal'], '收盤價'].astype(str)
    )


    ###########################################################################################
    # 🟢 強烈關注：趨勢 + 量 + MACD 三個全部符合才標記
    ###########################################################################################

    # ── 趨勢條件 ──
    w_trend_1 = df['MA_20'] > df['MA_50']                               # MA20 > MA50
    w_trend_2 = df['MA_20'] > df['MA_20'].shift(1)                      # MA20 斜率向上
    w_trend_3 = df['收盤價'] > df['MA_20']                               # 收盤站在 MA20 上方
    w_trend_ok = w_trend_1 & w_trend_2 & w_trend_3

    # ── 量條件 ──
    w_vol_1 = df['Volume_MA_5'] > df['Volume_MA_30']                    # 短量 > 長量
    w_vol_2 = df['Volume_MA_5'] > df['Volume_MA_5'].shift(1)            # 量能在放大
    w_vol_ok = w_vol_1 & w_vol_2

    # ── MACD 條件 ──
    w_macd_1 = df['MACD'] > 0                                           # MACD 站上零軸
    w_macd_2 = df['MACD_hist'] > 0                                      # 柱狀在正值
    w_macd_3 = df['MACD_hist'] > df['MACD_hist'].shift(1)               # 柱狀在擴大
    w_macd_ok = w_macd_1 & w_macd_2 & w_macd_3

    # ── 強烈關注：三個全部符合 ──
    df['Watch_Signal'] = w_trend_ok & w_vol_ok & w_macd_ok

    # ── 各條件逐日記錄（供最新一天面板顯示）──
    df['w_trend_1'] = w_trend_1
    df['w_trend_2'] = w_trend_2
    df['w_trend_3'] = w_trend_3
    df['w_vol_1']   = w_vol_1
    df['w_vol_2']   = w_vol_2
    df['w_macd_1']  = w_macd_1
    df['w_macd_2']  = w_macd_2
    df['w_macd_3']  = w_macd_3


    ###########################################################################################
    # 大多頭 / 大空頭 判斷（長期結構，加入 MA240）
    ###########################################################################################

    # ── 大多頭：四線多頭排列 + MA240 向上 + 量能 + MACD ──
    big_bull_price = (
        (df['收盤價'] > df['MA_20']) &
        (df['MA_20']  > df['MA_50']) &
        (df['MA_50']  > df['MA_240']) &
        (df['MA_240'] > df['MA_240'].shift(5))   # MA240 近5日向上
    )
    big_bull_vol  = df['Volume_MA_5'] > df['Volume_MA_80']
    big_bull_macd = (df['MACD'] > 0) & (df['MACD_hist'] > 0)

    df['BigBull'] = big_bull_price & big_bull_vol & big_bull_macd

    # ── 大空頭：四線空頭排列 + MA240 向下 + MACD ──
    big_bear_price = (
        (df['收盤價'] < df['MA_20']) &
        (df['MA_20']  < df['MA_50']) &
        (df['MA_50']  < df['MA_240']) &
        (df['MA_240'] < df['MA_240'].shift(5))   # MA240 近5日向下
    )
    big_bear_macd = (df['MACD'] < 0) & (df['MACD_hist'] < 0)

    df['BigBear'] = big_bear_price & big_bear_macd

    # ── 大趨勢標籤 ──
    df['BigTrend'] = np.select(
        [df['BigBull'], df['BigBear']],
        ['大多頭', '大空頭'],
        default='無明顯趨勢'
    )



    # ── 選股條件欄位 ──
    df['cond_ma5_gt_ma20'] = df['MA_5'] > df['MA_20']
    df['cond_ma5_gt_ma10'] = df['MA_5'] > df['MA_10']
    df['cond_vol_active'] = df['量短5長30比'] > 1
    df['cond_price_gt_ma20'] = df['收盤價'] > df['MA_20']

    _t1 = df['MA_20_斜率'] > 0
    _t2 = df['MA_20'] > df['MA_50']
    _t3 = (df['MACD'] > 0) & (df['MACD_hist'] > 0)
    _t4 = df['HH'] & df['HL']
    _trend_score = _t1.astype(int) + _t2.astype(int) + _t3.astype(int) + _t4.astype(int)
    df['cond_trend_bull'] = _trend_score >= 3

    df['my_signal'] = (
        df['cond_ma5_gt_ma10'] &
        df['cond_vol_active'] &
        df['cond_price_gt_ma20'] &
        df['cond_trend_bull']
    )

    df['my_signal_3d'] = df['my_signal'].rolling(3).sum()
    df['my_signal_5d'] = df['my_signal'].rolling(5).sum()
    df['my_signal_10d'] = df['my_signal'].rolling(10).sum()


    ###########################################################################################
    # 綜合Flag 計分（供排序用）
    ###########################################################################################

    flag = pd.Series(0, index=df.index)

    # === 量能 Flag ===
    vol_cond_red    = (df['Volume_MA_5'] > df['Volume_MA_10'] * 1.1) & \
                      (df['量短5長30比'] >= 1) & \
                      (df['成交金額'] > df['成交金額平均'] * 1.2)
    vol_cond_orange = df['Volume_MA_5'] > df['Volume_MA_10']
    vol_cond_blue   = df['成交金額'] > df['Volume_MA_5'] * 1.3

    flag += np.where(vol_cond_red, 3, np.where(vol_cond_orange, 2, np.where(vol_cond_blue, 1, 0)))

    # === 均線斜率 Flag ===
    ma_cond_red    = (df['MA_5_斜率'] > 0.001) & (df['MA_10_斜率'] > 0.001) & (df['MA_20_斜率'] > 0.001)
    ma_cond_orange = (df['MA_5_斜率'] > 0.001) & (df['MA_10_斜率'] > 0.001)

    flag += np.where(ma_cond_red, 3, np.where(ma_cond_orange, 2, 0))

    # === MACD / KD Flag ===
    kd_cond_purple = (df['MACD_hist_diff'] > 0) & (df['%K'] > 30) & (df['%K'] > df['%D'])
    kd_cond_blue_a = df['MACD'] > df['MACD-SL']
    kd_cond_blue_b = (df['MACD_hist_diff'] > 0) & (df['%K'] > 30)

    flag += np.where(kd_cond_purple, 3,
            np.where(kd_cond_blue_a, 2,
            np.where(kd_cond_blue_b, 1, 0)))

    df['綜合Flag'] = flag

    # 前三天的綜合Flag 分別放在不同欄位
    df['綜合Flag_1d'] = df['綜合Flag'].shift(1)
    df['綜合Flag_2d'] = df['綜合Flag'].shift(2)
    df['綜合Flag_3d'] = df['綜合Flag'].shift(3)

    return df

In [10]:
def full_technical_analysis(df):
    """
    統一處理流程：先進行 K棒分析，再執行市場分類與建議
    """
    df = analyze_candlestick(df)
    df = analyze_candlestick_multiday(df)
    df = apply_market_classification(df)
    return df

def analyze_candlestick(df):
    """
    根據 K 棒型態分析，標記常見型態並評估方向：
    - 長紅 / 長黑 K 棒、十字線、上下影線、多頭/空頭吞噬
    - 錘頭線 / 吊人線 / 流星線、下影長 > 實體兩倍
    """
    # 不 copy()：呼叫端已 copy，這裡直接改省一份記憶體
    _c = df['收盤價'].values
    _o = df['開盤價'].values
    _h = df['最高價'].values
    _l = df['最低價'].values

    body    = np.abs(_c - _o)
    candle  = _h - _l
    upper_b = np.maximum(_c, _o)   # 實體上緣
    lower_b = np.minimum(_c, _o)   # 實體下緣
    upper_s = _h - upper_b         # 上影
    lower_s = lower_b - _l         # 下影

    df['實體長度'] = body
    df['上影']     = upper_s
    df['下影']     = lower_s

    # 基本形態（numpy select 一次決定，不用三次 loc）
    candle_type = np.select(
        [
            (_c > _o) & (body > candle * 0.7),
            (_c < _o) & (body > candle * 0.7),
            body <= candle * 0.1,
        ],
        ['長紅K棒', '長黑K棒', '十字線'],
        default=''
    )
    df['K棒型態'] = candle_type

    # 上下影線（用 Series + 字串拼接）
    df.loc[upper_s > body, 'K棒型態'] += '|上影線長'
    df.loc[lower_s > body, 'K棒型態'] += '|下影線長'

    # 吞噬形態（np.roll 取昨日，比 shift+中間欄位快）
    _c1 = np.roll(_c, 1); _o1 = np.roll(_o, 1)
    yest_high = np.maximum(_c1, _o1)
    yest_low  = np.minimum(_c1, _o1)
    today_high = upper_b
    today_low  = lower_b

    df['昨收'] = _c1; df['昨開'] = _o1
    df['昨高'] = yest_high; df['昨低'] = yest_low
    df['今高'] = today_high; df['今低'] = today_low
    df.loc[(df['收盤價'] > df['開盤價']) & (df['昨收'] < df['昨開']) &
           (df['今高'] > df['昨高']) & (df['今低'] < df['昨低']), 'K棒型態'] += '|多頭吞噬'
    df.loc[(df['收盤價'] < df['開盤價']) & (df['昨收'] > df['昨開']) &
           (df['今高'] > df['昨高']) & (df['今低'] < df['昨低']), 'K棒型態'] += '|空頭吞噬'

    # 錘頭 / 吊人 / 流星
    df.loc[
        (df['實體長度'] < (df['最高價'] - df['最低價']) * 0.3) &
        (df['下影'] > df['實體長度'] * 2) &
        (df['上影'] < df['實體長度'] * 0.3), 'K棒型態'
    ] += '|錘頭線或吊人線'

    df.loc[
        (df['實體長度'] < (df['最高價'] - df['最低價']) * 0.3) &
        (df['上影'] > df['實體長度'] * 2) &
        (df['下影'] < df['實體長度'] * 0.3), 'K棒型態'
    ] += '|流星線'

    # K棒方向標註
    df['K棒方向'] = '中性'
    df.loc[df['K棒型態'].str.contains('長紅K棒|下影線長|多頭吞噬|錘頭線'), 'K棒方向'] = '正向'
    df.loc[df['K棒型態'].str.contains('長黑K棒|上影線長|空頭吞噬|流星線|吊人線'), 'K棒方向'] = '負向'
    df.loc[df['K棒型態'].str.contains('十字線'), 'K棒方向'] = '觀望'

    # ✅ 新增：下影線 > 實體長度 * 2
    df['K棒續強確認'] = ''
    df.loc[df['下影'] > df['實體長度'] * 2, 'K棒續強確認'] = '下影大於實體兩倍'

    return df

def analyze_candlestick_multiday(df):
    """
    判斷三日 K 棒型態（如晨星、暮星、三白兵、三隻烏鴉）與方向
    """
    df['多日K棒型態'] = ''
    df['多日K棒方向'] = ''

    df['三白兵'] = (
        (df['收盤價'] > df['開盤價']) &
        (df['收盤價'].shift(1) > df['開盤價'].shift(1)) &
        (df['收盤價'].shift(2) > df['開盤價'].shift(2)) &
        (df['收盤價'] > df['收盤價'].shift(1)) &
        (df['收盤價'].shift(1) > df['收盤價'].shift(2))
    )
    df.loc[df['三白兵'], ['多日K棒型態', '多日K棒方向']] = ['|三白兵', '正向']

    df['三隻烏鴉'] = (
        (df['收盤價'] < df['開盤價']) &
        (df['收盤價'].shift(1) < df['開盤價'].shift(1)) &
        (df['收盤價'].shift(2) < df['開盤價'].shift(2)) &
        (df['收盤價'] < df['收盤價'].shift(1)) &
        (df['收盤價'].shift(1) < df['收盤價'].shift(2))
    )
    df.loc[df['三隻烏鴉'], ['多日K棒型態', '多日K棒方向']] = ['|三隻烏鴉', '負向']

    df['晨星'] = (
        (df['收盤價'].shift(2) < df['開盤價'].shift(2)) &
        (abs(df['收盤價'].shift(1) - df['開盤價'].shift(1)) < (df['最高價'].shift(1) - df['最低價'].shift(1)) * 0.1) &
        (df['收盤價'] > df['開盤價']) &
        (df['收盤價'] > (df['收盤價'].shift(2) + df['開盤價'].shift(2)) / 2)
    )
    df.loc[df['晨星'], ['多日K棒型態', '多日K棒方向']] = ['|晨星', '正向']

    df['暮星'] = (
        (df['收盤價'].shift(2) > df['開盤價'].shift(2)) &
        (abs(df['收盤價'].shift(1) - df['開盤價'].shift(1)) < (df['最高價'].shift(1) - df['最低價'].shift(1)) * 0.1) &
        (df['收盤價'] < df['開盤價']) &
        (df['收盤價'] < (df['收盤價'].shift(2) + df['開盤價'].shift(2)) / 2)
    )
    df.loc[df['暮星'], ['多日K棒型態', '多日K棒方向']] = ['|暮星', '負向']

    return df

def apply_market_classification(df):
    import pandas as pd

    df['Market_State'] = ''
    df.loc[(df['收盤價'] < df['開盤價']) & ((df['最高價'] - df['開盤價']) / df['開盤價'] > 0.03), 'Market_State'] += ',衝高回落'
    df.loc[(df['收盤價'] > df['前日收盤價']) & (df['RSI'] < 30), 'Market_State'] += ',低檔翻揚'
    df.loc[(df['收盤價'] > df['MA_20'] * 0.98) & (df['收盤價'] < df['MA_20'] * 1.02) & (df['RSI'].between(45, 55)), 'Market_State'] += ',盤整震盪'
    df.loc[(df['MA_5'] > df['MA_20']) & (df['MA_20'] > df['MA_50']), 'Market_State'] += ',多頭排列'
    df.loc[(df['MA_5'] < df['MA_20']) & (df['MA_20'] < df['MA_50']), 'Market_State'] += ',空頭排列'
    df.loc[(df['收盤價'] > df['前日收盤價']) & (df['成交金額'] > df['Volume_MA_10'] * 1.5), 'Market_State'] += ',爆量上攻'
    df['Market_State'] = df['Market_State'].str.lstrip(',')

    # 預先建好 boolean mask，避免同一欄位重複 str.contains
    _ms = df['Market_State']
    _has_bull  = _ms.str.contains('多頭排列', regex=False)
    _has_bear  = _ms.str.contains('空頭排列', regex=False)
    _has_range = _ms.str.contains('盤整震盪', regex=False)
    _has_surge = _ms.str.contains('爆量上攻', regex=False)
    _macd_death = (df['MACD'] < df['MACD-SL']) & (df['MACD'].shift(1) >= df['MACD-SL'].shift(1))
    _macd_gold  = (df['MACD'] > df['MACD-SL']) & (df['MACD'].shift(1) <= df['MACD-SL'].shift(1))

    df['Buy_Signal'] = np.where(
        _has_bull & df['KD_golden_cross'] & _has_surge,
        '建議關注買點', ''
    )
    df['Sell_Signal'] = np.where(
        _has_bear & _macd_death & (df['收盤價'] < df['MA_20']),
        '建議留意風險', ''
    )
    df['Watch_Signal'] = np.where(
        _has_range & ~df['KD_golden_cross'] & ~df['RSI_rebound'] & ~_macd_gold,
        '觀望為宜', ''
    )
    df['RSI_diff'] = df['RSI'].diff()
    df['Reversal_Signal'] = np.where(
        _has_bear & (df['RSI'] < 30) & (df['RSI_diff'] > 5),
        '可能出現反轉訊號', ''
    )

    df['Action_Advice'] = ''
    df['Advice_Score'] = 0
    df.loc[df['Watch_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['觀望為宜', 1]
    df.loc[df['Buy_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['建議關注買點', 2]
    df.loc[df['Reversal_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['可能出現反轉訊號', 3]
    df.loc[df['Sell_Signal'] != '', ['Action_Advice', 'Advice_Score']] = ['建議留意風險', 4]

    df['Full_Summary'] = (
        '<b>分類：</b>' + df['Market_State'].fillna('') + '<br>' +
        '<b>建議：</b>' + df['Action_Advice'].fillna('') + '<br>' +
        '<b>評分：</b>' + df['Advice_Score'].astype(str) + '<br>' +
        '<b>K棒方向：</b>' + df.get('K棒方向', pd.Series([''] * len(df))) + '<br>' +
        '<b>多日K棒方向：</b>' + df.get('多日K棒方向', pd.Series([''] * len(df))) + '<br>' +
        '<b>K棒續強：</b>' + df.get('K棒續強確認', pd.Series([''] * len(df))) + '<br>' +
        '<b>多日K棒型態：</b>' + df.get('多日K棒型態', pd.Series([''] * len(df)))
    )
    
    df['GoDown'] = np.where(
        (df['成交金額'] > df['Volume_MA_10']) & (df['收盤價'] < df['開盤價']),
        '下跌帶量', ''
    )
    
    
    is_range = df['Market_State'].str.contains('盤整震盪', na=False)
    df['盤整震盪_前15日次數'] = (is_range.rolling(window=15, min_periods=1).sum())
    
    is_range2 = df['Market_State'].str.contains('多頭排列', na=False)
    df['多頭排列_前5日次數'] = (is_range2.rolling(window=5, min_periods=1).sum())
    
    is_range3 = df['GoDown'] == '下跌帶量'
    df['下跌帶量_前15日次數'] = (is_range3.rolling(window=15, min_periods=1).sum())
    
    is_range4 = df['MA_break']
    df['MA_break_前15日次數'] = (is_range4.rolling(window=15, min_periods=1).sum())

    
    is_range5 = df['MACD_bottom_signal']
    df['抄底訊號_前15日次數'] = (is_range5.rolling(window=15, min_periods=1).sum())
    
    is_range6 = df['voc_cross']
    df['量起點_前15日次數'] = (is_range6.rolling(window=15, min_periods=1).sum())
    
    
    # MACD 黃金交叉（boolean）
    golden_cross = (
        (df['MACD'] > df['MACD-SL']) &
        (df['MACD'].shift(1) <= df['MACD-SL'].shift(1))
    )

    # 轉成 0/1 再 rolling 計數
    df['黃金交叉_前15日次數'] = (
        golden_cross
        .astype(int)
        .rolling(window=15, min_periods=1)
        .sum()
    )
    
    
    
    is_range7 =(df['收盤價'] > df['MA_5'])
    df['價過5日_前15日次數'] = (is_range7.rolling(window=15, min_periods=1).sum())
    df['價過5日_avg'] =  df['價過5日_前15日次數'].mean()
    df['價過5日_前15日次數'] = np.where((df['價過5日_前15日次數'] < df['價過5日_avg']) |
                                  (df['Volume_MA_10'] < df['Volume_MA_50']) 
                                 ,0
                                 ,df['價過5日_前15日次數'])
    
    
    # checkpoint 搭配 is_range8
    df['checkpoint'] = False

    df.loc[
        (df['價過5日_前15日次數'] > 0) &
        (df['量短5長30比'] > 0) &
        (df['價過5日_前15日次數'] >= df['價過5日_前15日次數'].shift(1)) &
        (
            (df['量短5長30比'].shift(2) == 0) |
            (df['量短5長30比'].shift(3) == 0) |
            (df['量短5長30比'].shift(4) == 0) |
            (df['量短5長30比'].shift(5) == 0) |
            (df['量短5長30比'].shift(6) == 0)
        ) &
        (df['收盤價'] > df['MA_80']),
        'checkpoint'
    ] = True

    is_range8 = df['checkpoint']
    df['checkpoint_前10日次數'] = (is_range8.rolling(window=10, min_periods=1).sum())
    df['checkpoint_前5日次數'] = (is_range8.rolling(window=5, min_periods=1).sum())
    df['checkpoint_前2日次數'] = (is_range8.rolling(window=2, min_periods=1).sum())
    df['checkpoint_前1日次數'] = (is_range8.rolling(window=1, min_periods=1).sum())
        
        
    return df


In [11]:

def add_bollinger_bands(df, period=20, std_num=2):

    df = df.copy()

    # =========================
    # 布林線
    # =========================
    df['BB_MID'] = df['收盤價'].rolling(period).mean()

    std = df['收盤價'].rolling(period).std()

    df['BB_UPPER'] = df['BB_MID'] + std * std_num
    df['BB_LOWER'] = df['BB_MID'] - std * std_num


    # =========================
    # 布林寬度
    # =========================
    df['BB_WIDTH'] = ((df['BB_UPPER'] - df['BB_LOWER']) / df['BB_MID'])
    
    # =========================
    # 斜率
    # =========================
    df['BB_MID_斜率'] = df['BB_MID'].pct_change()

    df['BB_UPPER_斜率'] = df['BB_UPPER'].pct_change()

    df['BB_WIDTH_斜率'] = df['BB_WIDTH'].pct_change()

    # =========================
    # 平均布林寬度
    # =========================
    df['BB_WIDTH_avg'] = df['BB_WIDTH'].mean()

    # =========================
    # 動態平均
    # =========================
    df['BB_WIDTH_MA20'] = (df['BB_WIDTH'] .rolling(20).mean() )
    df['BB_WIDTH_MA5'] = (df['BB_WIDTH'] .rolling(5).mean() )

    # =========================
    # 當前寬度 / 平均寬度
    # =========================
    df['BB_WIDTH_RATIO'] = (df['BB_WIDTH'] / df['BB_WIDTH_MA20'])
  
    
    bbw_cross = (
        
        
        (df['BB_WIDTH'] > df['BB_WIDTH_MA20'])
       # & (df['BB_WIDTH'].shift(1) <= df['BB_WIDTH_MA20'].shift(1))
        & (df['收盤價'] > df['MA_20'])
    )

    ma5_cross = (
        (df['BB_WIDTH_MA5'] > df['BB_WIDTH_MA20'])
        &
        (df['BB_WIDTH_MA5'].shift(1) <= df['BB_WIDTH_MA20'].shift(1))
    )
    ma20_cross = (
        (df['收盤價'] > df['MA_20'])
        &
        (df['收盤價'].shift(1) <= df['MA_20'].shift(1))
    )

    
    #  df['MA_20_斜率'] > 0 &
    '''
    df['BB_golden_cross'] = (

        ( #(bbw_cross | ma5_cross) &
          (df['收盤價'] > df['MA_20'])
        ) 
       # |
        ma20_cross       
    )
    
    '''
    df['BB_golden_cross'] = (
        ma20_cross &
        df['MA_20_斜率'] > 0 
    )
    
    '''
    df['BB_golden_cross'] = ((df['BB_WIDTH'] > df['BB_WIDTH_MA20'])
                             & ((df['BB_WIDTH'].shift(1) <= df['BB_WIDTH_MA20'].shift(1)) |
                                (df['BB_WIDTH_MA5'].shift(1) <= df['BB_WIDTH_MA20'].shift(1))
                               )
                             & (df['收盤價'] > df['MA_20'])
                            )
  
    '''   
    df['bbw_cross'] = (bbw_cross
                            )
    
    df['BB_golden_cross_area'] = ((df['BB_WIDTH'] > df['BB_WIDTH_MA20'])
                            # & (df['BB_WIDTH'].shift(1) <= df['BB_WIDTH_MA20'].shift(1))
                            & (df['收盤價'] > df['MA_20'])
                            )
    
    
    df['BB_last_cross_date'] = df.loc[df['BB_golden_cross'], '年月日'].max()
    
    
    
    mask_up = (
        (df['MA_5_斜率'] > 0.001)
        & (df['MA_10_斜率'] > 0.001)
        & (df['MA_20_斜率'] > 0.001)
        & (
            df['BB_golden_cross']
            .rolling(5, min_periods=1)
            .max()
            .astype(bool)
        ))
    df['up']=(mask_up)
    df['up_last_cross_date'] = df.loc[df['up'], '年月日'].max()
    
    # =========================
    # 100個區間內在BB上軌的比例
    # =========================
    df['price_above_BB_UPPER'] = (df['收盤價'] > df['BB_UPPER']).astype(int)
    
    df['BB_UPPER_pct_long'] = df['price_above_BB_UPPER'].rolling(20).mean() * 100
    df['BB_UPPER_pct_short'] = df['price_above_BB_UPPER'].rolling(5).mean() * 100
    
    df['BB_UPPER_cross'] = ((df['BB_UPPER_pct_short'] > df['BB_UPPER_pct_long'])
                             & (df['BB_UPPER_pct_short'].shift(1) <= df['BB_UPPER_pct_long'].shift(1))
                            )
    df['BB_UPPER_cross_date'] = df.loc[df['BB_UPPER_cross'], '年月日'].max()
        
    return df




In [12]:
def save_plt_to_html(stock_number, stock_data, fig, text_area):
    try:
        os.makedirs("Html", exist_ok=True)
        filename = GetStockInfoByID(stock_number).replace('*', '')
        filepath = f"Html/[{stock_number}]{filename}.html"
        html = fig.to_html(full_html=True, include_plotlyjs='cdn')
        if text_area:
            html = html.replace("<head>", f"<head>\n{text_area}\n", 1)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"saved -> {filepath}")
        print(" Visit: http://localhost/IT//" + filepath)
    except Exception as e:
        print(f"{stock_number} error: {e}")


In [13]:
# 找到所有 Upper -> Down 的時間段
def find_intervals(upper, down):
    intervals = []
    for u in upper:
        for d in down:     
            if d > u:
                intervals.append([u, d])
                break
    return intervals

# 合併重維時間段，同時處理重複的標記
def merge_intervals(intervals):
    if not intervals:
        return intervals
    
    # 按每個區間的开始日期排序
    intervals.sort(key=lambda x: x[0])
    merged = []

    for current in intervals:
        if not merged:
            merged.append(current)
        else:
            last = merged[-1]
            if current[0] <= last[1] and current[1] > last[1]:
                # 分割出重複部分，並新增一個新區間
                merged.append([current[0], last[1]])
                merged.append([last[1], current[1]])
            elif current[0] > last[1]:
                merged.append(current)
            else:
                # 更新最後一個區間的結束日期
                last[1] = max(last[1], current[1])

    return merged

In [14]:
def get_highlight(stock_data):
    try:

        stock_data['highlight']=False

        macd_death_crosses = stock_data[
                (stock_data['MACD'] < stock_data['MACD-SL']) &  # 當前 MACD 線低於信號線
                (stock_data['MACD'].shift(1) >= stock_data['MACD-SL'].shift(1))  # 前一日 MACD 線高於或等於信號線
            ]
        #
        key_area = stock_data[
            #(stock_data['MA_5'] > stock_data['MA_20']) &
            (stock_data['MA_5'] > stock_data['MA_10']) &
            #(stock_data['%K'] > stock_data['%D']) &
            #(stock_data['MACD-SL'] > 0) &
             (stock_data['MA_20'] > stock_data['MA_80']) &
            #(stock_data['MA_50'] > stock_data['MA_80']) &

            (stock_data['MACD_hist_diff'] > 0) &
            (stock_data['MACD-SL'] > 0) &
            (stock_data['MACD'] > stock_data['MACD-SL']) 
        ]
        #
        Upper= list(key_area['年月日'])
        Down =list(macd_death_crosses['年月日'])

        Upper.sort()
        Down.sort()

        Down.append(pd.Timestamp(datetime.today().date() + timedelta(days=1)))
        intervals = find_intervals(Upper, Down)
        merged_intervals = merge_intervals(intervals)

        temp_dates =stock_data['年月日'] 
    
        '''
        try:
            intervals_1 =merged_intervals[::-1][0:2][0]
            #前一個區間日
            stock_data['highlight_stardate_1'] =intervals_1[0]
            stock_data['highlight_enddate_1'] = intervals_1[1]
            stock_data['highlight_stardate收盤價_1']=stock_data[stock_data['年月日']==intervals_1[0]]['收盤價'].iloc[0]
            stock_data['highlight_enddate收盤價_1']=stock_data[stock_data['年月日']==intervals_1[1]]['收盤價'].iloc[0]

        except Exception as error:
            print("")
           # print(f'Error get_highlight  前n個區間日 processing : {error}')      
        
        try:
            intervals_2 =merged_intervals[::-1][0:2][1]
            #前二個區間日
            stock_data['highlight_stardate_2'] =intervals_2[0]
            stock_data['highlight_enddate_2'] = intervals_2[1]
            stock_data['highlight_stardate收盤價_2']=stock_data[stock_data['年月日']==intervals_2[0]]['收盤價'].iloc[0]
            stock_data['highlight_enddate收盤價_2']=stock_data[stock_data['年月日']==intervals_2[1]]['收盤價'].iloc[0]

        except Exception as error:
            print("")
            #print(f'Error get_highlight2  前n個區間日 processing : {error}')
        '''
        
        # 向量化：建 Series 再一次 assign，比逐區間 loc 快
        _td = temp_dates.values
        _hl_flag    = np.zeros(len(stock_data), dtype=bool)
        _hl_date    = np.empty(len(stock_data), dtype=object)
        _hl_close   = np.full(len(stock_data), np.nan)
        _hl_enddate = np.empty(len(stock_data), dtype=object)

        # 預先建 date→close 的查找 dict
        _date_close = dict(zip(stock_data['年月日'].values, stock_data['收盤價'].values))

        for _start, _end in merged_intervals:
            _mask = (_td >= _start) & (_td <= _end)
            _hl_flag[_mask]    = True
            _hl_date[_mask]    = _start
            _hl_close[_mask]   = _date_close.get(_start, np.nan)
            _hl_enddate[_mask] = _end

        stock_data['highlight']        = _hl_flag
        stock_data['highlight_date']   = _hl_date
        stock_data['highlight_收盤價']  = _hl_close
        stock_data['highlight_enddate'] = _hl_enddate
        
    except Exception as error:
            print(f'【要追】Error get_highlight processing : {error}')
        
       
    
    return stock_data,merged_intervals

In [15]:
# ─── 向量化 vrect 輔助函式 ───────────────────────────────────────────
def _add_vrects_by_state(fig, dates, states, color_fn, row, col=1, **extra_kwargs):
    """
    對 states Series 找切換點，批次呼叫 add_vrect，取代 iterrows 迴圈。
    color_fn(state) -> fillcolor str
    """
    dates  = list(dates)
    states = list(states)
    n      = len(states)
    if n == 0:
        return fig

    segs = []   # (x0, x1, state)
    seg_start = dates[0]
    prev_state = states[0]
    for i in range(1, n):
        if states[i] != prev_state:
            segs.append((seg_start, dates[i], prev_state))
            seg_start  = dates[i]
            prev_state = states[i]
    segs.append((seg_start, dates[-1], prev_state))

    for x0, x1, st in segs:
        fc = color_fn(st)
        if fc:
            fig.add_vrect(x0=x0, x1=x1, fillcolor=fc, opacity=1,
                          line_width=0, row=row, col=col, **extra_kwargs)
    return fig


def _add_vrects_boolean(fig, dates, mask, fillcolor, line_color, row, col=1):
    """對 bool Series 找 True 連續段，批次 add_vrect。"""
    dates = list(dates)
    mask  = list(mask)
    n     = len(mask)
    if n == 0:
        return fig

    in_seg    = False
    seg_start = None
    for i in range(n):
        if mask[i] and not in_seg:
            seg_start = dates[i]
            in_seg    = True
        elif not mask[i] and in_seg:
            fig.add_vrect(x0=seg_start, x1=dates[i],
                          fillcolor=fillcolor, opacity=1,
                          line_width=1, line_color=line_color,
                          row=row, col=col)
            in_seg = False
    if in_seg and seg_start is not None:
        fig.add_vrect(x0=seg_start, x1=dates[-1],
                      fillcolor=fillcolor, opacity=1,
                      line_width=1, line_color=line_color,
                      row=row, col=col)
    return fig

# ─────────────────────────────────────────────────────────────────────
# gen_html 子函式群
# ─────────────────────────────────────────────────────────────────────

def _build_price_chart(fig, stock_data, merged_intervals):
    """Row 1：股價主圖"""

    # K棒（預設隱藏，hover 用）
    fig.add_trace(go.Candlestick(
        x=stock_data['年月日'],
        open=stock_data['開盤價'], high=stock_data['最高價'],
        low=stock_data['最低價'], close=stock_data['收盤價'],
        name='箱型圖', visible='legendonly',
        increasing_line_color='#00ffe7', decreasing_line_color='#ff4d6d',
        increasing_fillcolor='rgba(0,255,231,0.15)',
        decreasing_fillcolor='rgba(255,77,109,0.15)'
    ), row=1, col=1)

    # 股價主線
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['收盤價'],
        mode='lines', line=dict(color='#e8f4f4', width=2),
        text=stock_data['Full_Summary'], name='股價'
    ), row=1, col=1)

    # 布林帶（填色極淡）
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['BB_UPPER'],
        mode='lines', line=dict(width=0.8, color='rgba(0,255,231,0.25)'),
        name='BB UPPER上軌'
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['BB_MID'],
        mode='lines', line=dict(width=0.8, color='rgba(0,255,231,0.15)'),
        fill='tonexty', fillcolor='rgba(0,255,231,0.04)', name='BB MID中軌'
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['BB_LOWER'],
        mode='lines', line=dict(width=0.8, color='rgba(0,255,231,0.25)'),
        fill='tonexty', fillcolor='rgba(0,255,231,0.04)',
        name='BB LOWER下軌', visible='legendonly'
    ), row=1, col=1)

    # MA 線（短週期預設隱藏，減少視覺雜訊）
    ma_cfg = [
        ('MA_5',  1,   'rgba(0,255,231,0.55)',   True),
        ('MA_10', 1,   'rgba(0,220,200,0.55)',   True),
        ('MA_20', 1.5, 'rgba(0,200,180,0.75)',   True),
        ('MA_50', 1.5, 'rgba(0,170,155,0.75)',   True),
        ('MA_80', 2,   'rgba(100,220,210,0.85)', False),
        ('MA_240',2,   'rgba(200,120,255,0.7)',  False),
    ]
    for col_name, lw, color, hidden in ma_cfg:
        vis = 'legendonly' if hidden else True
        fig.add_trace(go.Scatter(
            x=stock_data['年月日'], y=stock_data[col_name],
            mode='lines', visible=vis,
            line=dict(width=lw, color=color),
            name=f'MA {col_name}'
        ), row=1, col=1)

    # 支撐線（橘黃系）
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Low_15d'],
        mode='lines', line=dict(width=1, dash='dot', color='#f0a500'),
        name='Low_15d'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Low_30d'],
        mode='lines', visible='legendonly',
        line=dict(width=1, dash='dot', color='#c87800'), name='Low_30d'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Low_60d'],
        mode='lines', visible='legendonly',
        line=dict(width=1, dash='dot', color='#9a5c00'), name='Low_60d'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Low_90d'],
        mode='lines', line=dict(width=2, dash='dash', color='#f0a500'),
        name='Low_90d'), row=1, col=1)

    # 壓力線（青藍系）
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['High_15d'],
        mode='lines', line=dict(width=1, dash='dot', color='#00c8e0'),
        name='High_15d'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['High_30d'],
        mode='lines', visible='legendonly',
        line=dict(width=1, dash='dot', color='#0096aa'), name='High_30d'), row=1, col=1)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['High_60d'],
        mode='lines', visible='legendonly',
        line=dict(width=1, dash='dot', color='#006878'), name='High_60d'), row=1, col=1)

    
    # 成交金額（secondary_y）
    _vol_colors = ['rgba(0,255,231,0.35)' if c == 'red' else 'rgba(255,77,109,0.25)'
                   for c in stock_data['Bar_Color']]
    fig.add_trace(go.Bar(
        x=stock_data['年月日'], y=stock_data['成交金額'],
        name='成交金額', marker_color=_vol_colors, opacity=1
    ), row=1, col=1, secondary_y=True)
    
    
    # 成交量
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['成交金額平均'],
        name='成交金額平均',
        mode='lines', line=dict(color='rgba(255,220,50,0.6)', width=1, dash='dot')
    ), row=1, col=1, secondary_y=True)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_5'],
        mode='lines', line=dict(width=2,color='#00c8e0'),
        name='成交量MA5'
    ), row=1, col=1, secondary_y=True)
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_10'],
        mode='lines', line=dict(width=2, dash='dot',  color='orange'),
        name='成交量MA10'
    ), row=1, col=1, secondary_y=True)    
    
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_30'],
        mode='lines', line=dict(width=2, dash='dash',  color='orange'),
        name='成交量MA30'
    ), row=1, col=1, secondary_y=True)    
    
    
    # 量
    y_val = (stock_data['量短5長30比'] * 50) + 100
    mask = y_val > 100
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'][mask], y=stock_data[mask]['成交金額平均'],
        name='>100', mode='markers',
        marker=dict(symbol='star', size=8, color='red')
    ), row=1, col=1, secondary_y=True)    
    
    
        
    # ── StateScore 主圖背景色帶（向量化）──
    def _state_bg(state):
        if state == '強勢': return 'rgba(29,158,117,0.15)'
        if state == '盤整': return 'rgba(180,160,80,0.06)'
        return 'rgba(160,40,40,0.09)'

    fig = _add_vrects_by_state(fig,
        stock_data['年月日'], stock_data['MarketState'],
        _state_bg, row=1)

    # 只標最後一個強勢區的現況標記
    if merged_intervals:
        x0, x1 = merged_intervals[-1]
        fig.add_vrect(
            x0=x0, x1=x1,
            annotation_text='強勢區(現)', annotation_position='top left',
            annotation_font=dict(color='rgba(0,255,231,0.7)', size=9),
            fillcolor='rgba(0,0,0,0)', opacity=1,
            line_width=1, line_color='rgba(0,255,231,0.3)', row=1, col=1
        )

    # ── 🟢 強烈關注色帶（向量化）──
    fig = _add_vrects_boolean(fig,
        stock_data['年月日'], stock_data['Watch_Signal'],
        fillcolor='rgba(0,255,160,0.13)',
        line_color='rgba(0,255,160,0.4)', row=1)


    # ── 大多頭 / 大空頭 色帶（向量化）──
    def _big_trend_color(st):
        if st == '大多頭': return 'rgba(0,200,100,0.07)'
        if st == '大空頭': return 'rgba(200,40,40,0.07)'
        return ''   # 無明顯趨勢不填色

    fig = _add_vrects_by_state(fig,
        stock_data['年月日'], stock_data['BigTrend'],
        _big_trend_color, row=1)

    # MA_break（預設隱藏）
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['MA_break']]['年月日'],
        y=stock_data[stock_data['MA_break']]['MA_20'],
        mode='markers', marker_symbol='star',
        marker_color='#ff4d6d', marker_size=12,
        name='MA_break', visible='legendonly'
    ), row=1, col=1)

    # ── 選股條件全滿足：黃色發光圓點 ──
    _sig = stock_data[stock_data.get('my_signal', False) == True] if 'my_signal' in stock_data.columns else stock_data.iloc[0:0]
    fig.add_trace(go.Scatter(
        x=_sig['年月日'],
        y=_sig['收盤價'],
        mode='markers',
        marker=dict(
            symbol='circle',
            size=14,
            color='#ffee00',
            line=dict(color='#ffee00', width=2),
            opacity=0.95,
        ),
        name='選股訊號',
        hovertemplate='<b>選股訊號</b><br>%{x}<br>收盤價: %{y}<extra></extra>'
    ), row=1, col=1)

    

    # 三白兵（預設隱藏）
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['三白兵']]['年月日'],
        y=stock_data[stock_data['三白兵']]['收盤價'],
        mode='markers', marker_symbol='star',
        marker_color='#ffe566', marker_size=14,
        name='三白兵', visible='legendonly'
    ), row=1, col=1)

    # 價過5日標記（預設隱藏）
    y_val = stock_data['價過5日_前15日次數'] / 10
    mask = y_val > 0
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'][mask], y=stock_data[mask]['Low_15d'],
        name='【價】', mode='markers',
        marker=dict(symbol='star', size=8, color='#ff4d6d'),
        visible='legendonly'
    ), row=1, col=1)
    

    # bbw_cross
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['bbw_cross']]['年月日'],
        y=stock_data[stock_data['bbw_cross']]['BB_UPPER'] ,
        mode='markers', marker_color='red',
        marker_size=8, name='bbw_cross', 
    ), row=1, col=1)
    
    
    # BB_golden_cross
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['BB_golden_cross']]['年月日'],
        y=stock_data[stock_data['BB_golden_cross']]['收盤價'] ,
        mode='markers', marker_color='rgba(232, 236, 63,0.5)',
        marker_size=20, name='BB_golden_cross', 
    ), row=1, col=1)
    
    # MA斜率三線同步向上
    mask = (
         (stock_data['MA_5_斜率'] > 0.001)
        & (stock_data['MA_10_斜率'] > 0.001)
        & (stock_data['MA_20_斜率'] > 0.001)
    )

    fig.add_trace(go.Scatter(x=stock_data['年月日'][mask], 
                             y=stock_data['MA_20'][mask],
                             mode='markers',
                             marker=dict( symbol='triangle-up', size=10,color='lime'),
                             name='BB_golden_cross_MA_UP'
                            ), row=1, col=1)
    
    mask1 = (
        (stock_data['MA_5_斜率'] > 0.001)
        & (stock_data['MA_10_斜率'] > 0.001)
        & (stock_data['MA_20_斜率'] > 0.001)
        & (
            stock_data['BB_golden_cross']
            .rolling(5, min_periods=1)
            .max()
            .astype(bool)
        ))
    fig.add_trace(go.Scatter(x=stock_data['年月日'][mask1], 
                             y=stock_data['MA_20'][mask1],
                             mode='markers',
                             marker=dict( symbol='star', size=10,color='yellow'),
                             name='UP_UP'
                            ), row=1, col=1)
    
    
    

    return fig


def _build_macd_chart(fig, stock_data, macd_golden_crosses, macd_death_crosses):
    """Row 2：MACD 子圖（KD 已移至 Row 3，此處不放 KD）"""
    i_row = 2

    # MACD 柱狀（正橙負藍）
    macd_hist = stock_data['MACD_hist']
    macd_colors = ['#f6b26b' if v >= 0 else '#9fc5e8' for v in macd_hist]
    fig.add_trace(go.Bar(
        x=stock_data['年月日'], y=macd_hist,
        marker_color=macd_colors, name='MACD 柱狀', opacity=0.6
    ), row=i_row, col=1)

    # MACD 線 & 信號線
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['MACD'],
        mode='lines', line=dict(color='#ebbd67', width=1.5),
        name='Diff(12,26)'
    ), row=i_row, col=1)
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['MACD-SL'],
        mode='lines', line=dict(color='#67ceeb', width=1.5),
        name='MACD(9)'
    ), row=i_row, col=1)

    # 參考線（預設隱藏）
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['MACD-SL_min'],
        mode='lines', line=dict(color='gray', width=0.8),
        visible='legendonly', name='MACD-SL_min'
    ), row=i_row, col=1)
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['MACD-SL_max'],
        mode='lines', line=dict(color='gray', width=0.8),
        visible='legendonly', name='MACD-SL_max'
    ), row=i_row, col=1)

    # 黃金/死亡交叉（三角，顏色統一：買綠賣紅）
    fig.add_trace(go.Scatter(
        x=macd_golden_crosses['年月日'], y=macd_golden_crosses['MACD'],
        mode='markers', marker_symbol='triangle-up',
        marker_color='lime', marker_size=10, name='MACD 黃金交叉'
    ), row=i_row, col=1)
    fig.add_trace(go.Scatter(
        x=macd_death_crosses['年月日'], y=macd_death_crosses['MACD'],
        mode='markers', marker_symbol='triangle-down',
        marker_color='#ff4d6d', marker_size=10, name='MACD 死亡交叉'
    ), row=i_row, col=1)

    # 抄底訊號（顯示）
    bottom_pts = stock_data[stock_data['MACD_bottom_signal']]
    fig.add_trace(go.Scatter(
        x=bottom_pts['年月日'], y=bottom_pts['MACD'],
        mode='markers', marker_symbol='star',
        marker_color='lime', marker_size=13, name='MACD 抄底訊號'
    ), row=i_row, col=1)

    # 逃頂訊號（預設隱藏）
    top_pts = stock_data[stock_data['MACD_top_signal']]
    fig.add_trace(go.Scatter(
        x=top_pts['年月日'], y=top_pts['MACD'],
        mode='markers', marker_symbol='star',
        marker_color='magenta', marker_size=13,
        name='MACD 逃頂訊號', visible='legendonly'
    ), row=i_row, col=1)

    # ── 大多頭 / 大空頭 色帶（向量化）──
    def _big_trend_color(st):
        if st == '大多頭': return 'rgba(0,200,100,0.07)'
        if st == '大空頭': return 'rgba(200,40,40,0.07)'
        return ''   # 無明顯趨勢不填色

    fig = _add_vrects_by_state(fig,
        stock_data['年月日'], stock_data['BigTrend'],
        _big_trend_color, row=1)

    # MA_break（預設隱藏）
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['MA_break']]['年月日'],
        y=stock_data[stock_data['MA_break']]['MACD'],
        mode='markers', marker_symbol='star',
        marker_color='red', marker_size=12,
        name='MA_break(MACD)', visible='legendonly'
    ), row=i_row, col=1)

    # TrendScore（預設隱藏）
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['TrendScore'],
        name='TrendStore', text=stock_data['TrendScorertxt'],
        mode='lines', line=dict(color='rgba(100,100,255,0.5)', width=1),
        visible='legendonly'
    ), row=i_row, col=1, secondary_y=True)

    # StateScore 折線（0-100，掛在 secondary_y，不影響 MACD 主軸）
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['StateScore'],
        name='StateScore', text=stock_data['StateScore_txt'],
        mode='lines', line=dict(color='rgba(127,119,221,0.6)', width=1, dash='dot'),
        visible=True
    ), row=i_row, col=1, secondary_y=True)

    fig.update_yaxes(title_text='MACD', row=i_row, col=1)
    fig.add_hline(y=0, line_dash='solid',
                  line_color='rgba(255,255,255,0.15)', line_width=1,
                  row=i_row, col=1)
    # StateScore secondary_y：固定 0-100（獨立於 MACD 主軸）
    fig.update_yaxes(
        range=[0, 100], showgrid=False,
        tickvals=[20, 90], ticktext=['20', '90'],
        tickfont=dict(size=9, color='rgba(127,119,221,0.6)'),
        title_text='Score',
        title_font=dict(size=9, color='rgba(127,119,221,0.5)'),
        row=i_row, col=1, secondary_y=True
    )
    # MACD 主軸讓 Plotly 自動縮放（不被 secondary 干擾）
    fig.update_yaxes(autorange=True, row=i_row, col=1, secondary_y=False)
    # StateScore 40/70 參考虛線（畫在 secondary_y 的值域）
    fig.add_hline(y=90, line_dash='dot',
                  line_color='rgba(29,158,117,0.3)', line_width=1,
                  row=i_row, col=1, secondary_y=True)
    fig.add_hline(y=20, line_dash='dot',
                  line_color='rgba(239,159,39,0.3)', line_width=1,
                  row=i_row, col=1, secondary_y=True)


    # ── 綜合Flag 折線（secondary_y，正規化至 0-100 與 StateScore 共用軸）──
    # 綜合Flag 原始 0-9，乘以 100/9 ≈ 11.1 對應到 0-100 軸
    if '綜合Flag' in stock_data.columns:
        flag_scaled = stock_data['綜合Flag'] * (100 / 9)
        fig.add_trace(go.Scatter(
            x=stock_data['年月日'],
            y=flag_scaled,
            name='綜合Flag（右軸）',
            mode='lines+markers',
            line=dict(color='rgba(255,200,0,0.8)', width=1.5, dash='dot'),
            marker=dict(size=4, color='rgba(255,200,0,0.9)'),
            visible=True,
            customdata=stock_data['綜合Flag'],
            hovertemplate='綜合Flag: %{customdata}/9<extra></extra>'
        ), row=2, col=1, secondary_y=True)

        # secondary_y 補上 Flag 刻度標示（與 StateScore 共用 0-100 軸）
        # 對應：3→33, 6→67, 9→100
        fig.update_yaxes(
            range=[0, 100], showgrid=False,
            tickvals=[20, 33, 67, 90, 100],
            ticktext=['20', 'F3', 'F6', '90', 'F9'],
            tickfont=dict(size=9, color='rgba(255,200,0,0.6)'),
            title_text='Score/Flag',
            title_font=dict(size=9, color='rgba(180,160,80,0.5)'),
            row=2, col=1, secondary_y=True
        )

    return fig


def _build_kd_chart(fig, stock_data):
    """Row 3：KD 獨立子圖（避免與 MACD 軸混亂）"""
    i_row = 3

    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['%K'],
        mode='lines', line=dict(width=1.5, color='#f0a500'), name='%K'
    ), row=i_row, col=1)
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['%D'],
        mode='lines', line=dict(width=1.5, color='#00c8e0'), name='%D'
    ), row=i_row, col=1)

        
    # KD 黃金交叉
    '''
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['KD_golden_cross']]['年月日'],
        y=stock_data[stock_data['KD_golden_cross']]['%D'],
        mode='markers', marker_symbol='star',
        marker_color='lime', marker_size=12, name='KD 黃金交叉'
    ), row=i_row, col=1)
    '''
    # 超買超賣參考線
    fig.add_hline(y=20, line_dash='dot',
                  line_color='rgba(255,77,109,0.4)',
                  annotation_text='超賣20',
                  annotation_font=dict(size=9, color='rgba(255,77,109,0.6)'),
                  row=i_row, col=1)
    fig.add_hline(y=80, line_dash='dot',
                  line_color='rgba(0,255,200,0.4)',
                  annotation_text='超買80',
                  annotation_font=dict(size=9, color='rgba(0,255,200,0.6)'),
                  row=i_row, col=1)

    fig.update_yaxes(range=[0, 100], title_text='KD', row=i_row, col=1)
    return fig


def _build_volume_chart(fig, stock_data):
    """Row 4：交易量子圖"""
    i_row = 4

    # ── 成交量柱狀圖（主體，漲紅跌綠）──
    _vol_colors = ['rgba(0,255,231,0.45)' if c == 'red' else 'rgba(255,77,109,0.35)'
                   for c in stock_data['Bar_Color']]
    fig.add_trace(go.Bar(
        x=stock_data['年月日'], y=stock_data['成交金額'],
        name='成交量', marker_color=_vol_colors, opacity=1
    ), row=i_row, col=1, secondary_y=True)

    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['成交金額'],
        visible='legendonly', mode='lines',
        line=dict(width=1, color='gray'), name='成交金額(線)'
    ), row=i_row, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_5'],
        mode='lines', line=dict(width=1, dash='dash', color='#4a90d9'),
        name='成交量MA_5', visible='legendonly'
    ), row=i_row, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_10'],
        mode='lines', line=dict(width=2, color='orange'),
        name='成交量MA10'
    ), row=i_row, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_30'],
        mode='lines', text=stock_data['分類'],
        line=dict(width=2, dash='dash', color='#ff4d6d'),
        name='成交量MA30', visible='legendonly'
    ), row=i_row, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_50'],
        mode='lines', line=dict(width=2, color='#cc3355'),
        name='成交量MA_50', visible='legendonly'
    ), row=i_row, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['Volume_MA_80'],
        mode='lines', line=dict(width=2, color='#00c8a0'),
        name='成交量MA_80', visible='legendonly'
    ), row=i_row, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(x=stock_data['年月日'], y=stock_data['成交金額平均'],
        name='成交金額平均', visible='legendonly',
        mode='lines', line=dict(color='gold', width=2)
    ), row=i_row, col=1, secondary_y=True)

    # 量起點
    voc_cross = stock_data[stock_data['voc_cross']]
    fig.add_trace(go.Scatter(
        x=voc_cross['年月日'], y=voc_cross['Volume_MA_5'],
        mode='markers', marker_symbol='triangle-up',
        marker_color='yellow', marker_size=12, name='量起點'
    ), row=i_row, col=1, secondary_y=True)

    # 量比指標
    
    y_val = (stock_data['量短5長30比'] * 50) + 100
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=y_val,
        name='【量】短5長30比',
        mode='lines', line=dict(color='rgba(150,150,150,0.5)', width=1)
    ), row=i_row, col=1)
    mask = y_val > 100
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'][mask], y=[100] * mask.sum(),
        name='>100', mode='markers',
        marker=dict(symbol='star', size=8, color='red')
    ), row=i_row, col=1)
    

    # ── 交易量圖：StateScore 色帶（向量化）──
    def _vol_state_color(state):
        if state == '強勢': return 'rgba(29,158,117,0.18)'
        if state == '盤整': return 'rgba(180,160,80,0.07)'
        return 'rgba(160,40,40,0.12)'

    fig = _add_vrects_by_state(fig,
        stock_data['年月日'], stock_data['MarketState'],
        _vol_state_color, row=4)

    fig.update_xaxes(title_text='成交量', row=i_row, col=1)
    return fig


def _build_bband_chart(fig, stock_data):
    """Row 5：BBand 軌道 + checkpoint"""
    i_row = 5
 
    
    para_bb_ = 10
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'],
        y=stock_data['BB_WIDTH_RATIO'] * para_bb_,
        mode='lines', name='WIDTH RATIO',visible='legendonly',
        line=dict(color='rgba(0,255,231,0.6)', width=1.5)
    ), row=i_row, col=1)
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['BB_golden_cross']]['年月日'],
        y=stock_data[stock_data['BB_golden_cross']]['BB_WIDTH_RATIO'] * para_bb_,
        mode='markers', marker_symbol='star',visible='legendonly',
        marker_color='#ff4d6d', marker_size=12, name='BB_golden_cross'
    ), row=i_row, col=1)
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['BB_golden_cross_area']]['年月日'],
        y=stock_data[stock_data['BB_golden_cross_area']]['BB_WIDTH_RATIO'] * para_bb_,
        mode='markers', marker_color='rgba(255,77,109,0.4)',visible='legendonly',
        marker_size=6, name='BB_golden_cross_area', 
    ), row=i_row, col=1)
    
    # 量
    y_val = (stock_data['量短5長30比'] * 50) + 100
    mask = y_val > 100
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'][mask], y=stock_data[mask]['BB_UPPER_pct_short'],
        name='>100', mode='markers',
        marker=dict(symbol='star', size=8, color='red')
    ), row=i_row, col=1, secondary_y=True)
    fig.add_trace(go.Scatter(
        x=stock_data[stock_data['BB_UPPER_cross']]['年月日'],
        y=stock_data[stock_data['BB_UPPER_cross']]['BB_UPPER_pct_long'] ,
        mode='markers', marker_color='rgba(232, 236, 63,0.5)',
        marker_size=20, name='BB_UPPER_cross', 
    ), row=i_row, col=1, secondary_y=True)
    
    

    # checkpoint
    checkpoint = stock_data[stock_data['checkpoint']]
    fig.add_trace(go.Scatter(
        x=checkpoint['年月日'], y=[0] * len(checkpoint),
        mode='markers', marker_symbol='triangle-up',
        marker_color='#00ffe7', marker_size=10, name='checkpoint',
    ), row=i_row, col=1)
    
    
    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['BB_UPPER_pct_short'],
        name='BB_UPPER_pct_short',
        mode='lines', line=dict(color='rgba(255,220,50,0.6)', width=1.5, dash='dot')
    ), row=i_row, col=1, secondary_y=True)

    fig.add_trace(go.Scatter(
        x=stock_data['年月日'], y=stock_data['BB_UPPER_pct_long'],
        name='BB_UPPER_pct_long',
        mode='lines', line=dict(color='rgba(50,220,255,0.6)', width=1.5, dash='dot')
    ), row=i_row, col=1, secondary_y=True)
        
        
    fig.update_xaxes(title_text='BBand軌道(布林通道)', row=i_row, col=1)
    return fig


def _apply_cyber_theme(fig, stock_data, stock_number):
    """科技風主題 + 所有軸統一設定"""
    _cyber_bg    = '#060d0d'
    _cyber_plot  = '#080f10'
    _cyber_grid  = 'rgba(0,255,231,0.07)'
    _cyber_text  = '#7ecece'
    _cyber_accent= '#00ffe7'
    _cyber_font  = 'Share Tech Mono, Courier New, monospace'

    # 壓低 row1 secondary_y 成交金額
    _vol_max = stock_data['成交金額'].max()
    fig.update_yaxes(range=[0, _vol_max * 5], row=1, col=1, secondary_y=True,
                     showticklabels=False, showgrid=False)

    fig.update_layout(
        paper_bgcolor=_cyber_bg,
        plot_bgcolor=_cyber_plot,
        font=dict(family=_cyber_font, color=_cyber_text, size=11),
        title=dict(
            text=(f'<b style="letter-spacing:3px">{stock_number}</b>'
                  f'  <span style="font-size:11px;color:#4a8a8a;letter-spacing:1px">'
                  f'TECHNICAL ANALYSIS</span>'),
            font=dict(family=_cyber_font, color=_cyber_accent, size=16),
            x=0.01, y=0.99, xanchor='left'
        ),
        legend=dict(
            bgcolor='rgba(5,15,15,0.85)',
            bordercolor=_cyber_accent, borderwidth=1,
            font=dict(family=_cyber_font, color=_cyber_text, size=10),
            orientation='v', groupclick='toggleitem',
        ),
        hoverlabel=dict(
            bgcolor='#040c0c', bordercolor=_cyber_accent,
            font=dict(family=_cyber_font, color=_cyber_accent, size=11)
        ),
        margin=dict(l=60, r=200, t=50, b=40),

        barmode='overlay',
    )

    _axis_common = dict(
        gridcolor=_cyber_grid,
        zerolinecolor='rgba(0,255,231,0.15)',
        tickfont=dict(family=_cyber_font, color=_cyber_text, size=10),
        linecolor='rgba(0,255,231,0.2)',
        showspikes=True, spikecolor=_cyber_accent,
        spikethickness=1, spikedash='dot',
    )
    for row_i in range(1, 6):
        fig.update_xaxes(**_axis_common, row=row_i, col=1)
        fig.update_yaxes(**_axis_common, row=row_i, col=1)
        fig.update_yaxes(**_axis_common, row=row_i, col=1, secondary_y=True)

    for ann in fig.layout.annotations:
        ann.font.color  = _cyber_accent
        ann.font.family = _cyber_font
        ann.font.size   = 11

    return fig


def _build_signal_panel(stock_data, stock_number):
    """右下角 SYS PANEL（修正 indicator_colors 數量對不上 bug）"""
    last = stock_data.iloc[-1]

    # ── 大多頭 / 大空頭 最新狀態 ──
    big_trend = last.get('BigTrend', '無明顯趨勢')
    if big_trend == '大多頭':
        big_trend_html = '<div style="font-size:13px;font-weight:500;color:#00ff88;letter-spacing:1px;margin-bottom:4px;">🟢 大多頭</div>'
    elif big_trend == '大空頭':
        big_trend_html = '<div style="font-size:13px;font-weight:500;color:#ff4466;letter-spacing:1px;margin-bottom:4px;">🔴 大空頭</div>'
    else:
        big_trend_html = '<div style="font-size:13px;color:#4a7a7a;letter-spacing:1px;margin-bottom:4px;">⬜ 無明顯趨勢</div>'

    # ── 最新一天各條件狀態 ──
    def _ck(col):
        v = last.get(col, False)
        return '<span style="color:#00ffe7;">✅</span>' if v else '<span style="color:#ff4466;">❌</span>'

    _t1_ok = bool((last.get('MA_20_斜率') or 0) > 0)
    _t2_ok = bool((last.get('MA_20') or 0) > (last.get('MA_50') or 0))
    _t3_ok = bool((last.get('MACD') or 0) > 0 and (last.get('MACD_hist') or 0) > 0)
    _t4_ok = bool(last.get('HH', False) and last.get('HL', False))
    _trend_count = sum([_t1_ok, _t2_ok, _t3_ok, _t4_ok])

    def _ck2(v):
        return '<span style="color:#00ffe7;">✅</span>' if v else '<span style="color:#ff4466;">❌</span>'

    my_signal_ok = bool(last.get('my_signal', False))
    signal_color = '#00ff88' if my_signal_ok else '#4a7a7a'
    signal_label = '🟢 條件全滿足' if my_signal_ok else '⬜ 條件不足'

    _watch_html = (
        f'<div style="font-size:12px;font-weight:500;color:{signal_color};margin-bottom:8px;">{signal_label}</div>'
        f'<div style="color:#4a7a7a;font-size:9px;margin-bottom:3px;">── 條件 ──</div>'
        f'<div>{_ck("cond_ma5_gt_ma20")} MA5 &gt; MA20</div>'
        f'<div>{_ck("cond_vol_active")} 有量（短長量比 &gt; 1）</div>'
        f'<div>{_ck("cond_price_gt_ma20")} 股價 &gt; MA20</div>'
        f'<div style="color:#4a7a7a;font-size:9px;margin:6px 0 3px;">── 趨勢多頭（{_trend_count}/4）──</div>'
        f'<div>{_ck2(_t1_ok)} MA20 斜率向上</div>'
        f'<div>{_ck2(_t2_ok)} MA20 &gt; MA50</div>'
        f'<div>{_ck2(_t3_ok)} MACD &gt; 0 且柱狀 &gt; 0</div>'
        f'<div>{_ck2(_t4_ok)} HH &amp; HL 成立</div>'
    )

    indicator_defs = [
        ('MACD 黃金交叉',    bool(last.get('MACD_golden_cross', False))),
        ('KD 黃金交叉',      bool(last.get('KD_golden_cross',   False))),
        ('MA 均線突破',      bool(last.get('MA_break',           False))),
        ('BB 擴張訊號',      bool(last.get('BB_golden_cross',    False))),
        ('Checkpoint 觀察點',bool(last.get('checkpoint',         False))),
    ]

    html_content = ''
    for label, is_active in indicator_defs:
        dot_color   = '#00ffe7' if is_active else '#1a3a3a'
        dot_glow    = '0 0 6px #00ffe7, 0 0 12px #00ffe7' if is_active else 'none'
        label_color = '#00ffe7' if is_active else '#4a7a7a'
        html_content += (
            f'<div style="display:flex;align-items:center;margin-bottom:6px;">'
            f'<div style="width:8px;height:8px;border-radius:50%;background:{dot_color};'
            f'box-shadow:{dot_glow};margin-right:10px;flex-shrink:0;"></div>'
            f'<span style="font-family:\'Courier New\',monospace;font-size:11px;'
            f'color:{label_color};letter-spacing:0.5px;">{label}</span></div>'
        )

    html_iframe = (
        '<style>'
        '@import url(\'https://fonts.googleapis.com/css2?family=Share+Tech+Mono&display=swap\');'
        '.cyber-panel-btn{display:inline-block;background:transparent;border:1px solid #00ffe7;'
        'color:#00ffe7;font-family:\'Share Tech Mono\',\'Courier New\',monospace;font-size:11px;'
        'letter-spacing:1px;padding:4px 10px;margin:2px;cursor:pointer;text-transform:uppercase;'
        'transition:background 0.2s,box-shadow 0.2s;text-decoration:none;}'
        '.cyber-panel-btn:hover{background:rgba(0,255,231,0.12);box-shadow:0 0 8px #00ffe7;color:#00ffe7;}'
        '</style>'
        f'<div style="font-family:\'Share Tech Mono\',\'Courier New\',monospace;">'
        f'<div id="iframe-container" style="height:750px;display:none;margin-bottom:6px;">'
        f'<iframe src="https://www.wantgoo.com/stock/{stock_number}" width="100%" height="100%" frameborder="0"></iframe></div>'
        f'<div id="iframe-container2" style="height:750px;display:none;margin-bottom:6px;">'
        f'<iframe src="https://www.wantgoo.com/stock/{stock_number}/institutional-investors/trend" width="100%" height="100%" frameborder="0"></iframe></div>'
        f'<div style="display:flex;flex-wrap:wrap;gap:2px;margin-bottom:4px;">'
        f'<button class="cyber-panel-btn" onclick="toggleIframe()">&#9654; 行情</button>'
        f'<button class="cyber-panel-btn" onclick="toggleIframe2()">&#9654; 法人動態</button></div>'
        f'<div style="display:flex;flex-wrap:wrap;gap:2px;">'
        f'<a class="cyber-panel-btn" href="https://www.wantgoo.com/stock/{stock_number}" target="_blank">玩股網</a>'
        f'<a class="cyber-panel-btn" href="https://tw.stock.yahoo.com/quote/{stock_number}.TW" target="_blank">Yahoo</a>'
        f'<a class="cyber-panel-btn" href="https://pscnetinvest.moneydj.com/z/zc/zca/zca.djhtm?a={stock_number}" target="_blank">MoneyDJ</a>'
        f'</div></div>'
        f'<script>'
        f'function toggleIframe(){{var c=document.getElementById(\'iframe-container\');c.style.display=c.style.display===\'none\'?\'block\':\'none\';}}'
        f'function toggleIframe2(){{var c=document.getElementById(\'iframe-container2\');c.style.display=c.style.display===\'none\'?\'block\':\'none\';}}'
        f'</script>'
    )

    text_area = (
        '<style>'
        '.text-area{position:fixed;bottom:16px;right:16px;'
        'background:linear-gradient(160deg,#050f0f 0%,#071a1a 100%);'
        'border:1px solid #00ffe7;'
        'box-shadow:0 0 18px rgba(0,255,231,0.25),inset 0 0 30px rgba(0,255,231,0.04);'
        'padding:14px 16px;z-index:1000;max-width:300px;word-wrap:break-word;'
        'font-family:\'Microsoft JhengHei\',\'PingFang TC\',\'Heiti TC\',sans-serif;'
        'font-size:11px;color:#7ecece;}'
        '.text-area::before{content:\'\';position:absolute;top:0;left:0;right:0;height:2px;'
        'background:linear-gradient(90deg,transparent,#00ffe7,transparent);}'
        '.text-area::after{content:\'\';position:absolute;inset:0;'
        'background:repeating-linear-gradient(0deg,transparent,transparent 2px,'
        'rgba(0,255,231,0.015) 2px,rgba(0,255,231,0.015) 4px);pointer-events:none;}'
        '</style>'
        f'<div class="text-area">'
        f'<div style="font-size:10px;line-height:1.9;">'
        + _watch_html +
        f'</div>'
        f'</div>'
    )

    return text_area


# ─────────────────────────────────────────────────────────────────────
# gen_html 主函式
# ─────────────────────────────────────────────────────────────────────
def gen_html(stock_number, stock_data):
    """
    主函式：拆分子函式，降低視覺雜訊，KD 獨立子圖，訊號燈全部正確
    變更摘要：
      - 拆成 5 個 sub 函式（price / macd / kd / volume / bband）
      - KD 從 MACD secondary_y 移至獨立 row3，解決雙軸混亂
      - row 比例: 0.42 / 0.18 / 0.12 / 0.16 / 0.12
      - 主圖訊號改為 MACD+KD+量 三重確認複合訊號（減少假訊號雜訊）
      - 次要訊號（三白兵/MA_break/BB等）改為 legendonly
      - highlight vrect 加上標籤文字
      - Signal Panel indicator_colors 修正（原本只有 1 個 color 但有 4 個 label）
      - xxx_save_plt_to_html 已移除
      - rangebreaks 補上 row4/5
    """
    N = 150
    stock_data = stock_data.tail(N).copy()

    macd_golden_crosses = stock_data[
        (stock_data['MACD'] > stock_data['MACD-SL']) &
        (stock_data['MACD'].shift(1) <= stock_data['MACD-SL'].shift(1))
    ]
    macd_death_crosses = stock_data[
        (stock_data['MACD'] < stock_data['MACD-SL']) &
        (stock_data['MACD'].shift(1) >= stock_data['MACD-SL'].shift(1))
    ]
    stock_data, merged_intervals = get_highlight(stock_data)
    
    
    # 5 列子圖（新增 KD 獨立列）
    pic=3
    fig = subplots.make_subplots(
        rows=pic, cols=1,
        subplot_titles=('股價與移動平均線', 'MACD 指標', 'KD 指標', '交易量(有量)', 'BBand軌道'),
        shared_xaxes=True,
        vertical_spacing=0.025,
        specs=[[{"secondary_y": True}] for _ in range(pic)],
        row_heights=[0.66, 0.22, 0.12]
    )

    fig = _build_price_chart(fig, stock_data, merged_intervals)
    fig = _build_macd_chart(fig, stock_data, macd_golden_crosses, macd_death_crosses)
    fig = _build_kd_chart(fig, stock_data)
    #fig = _build_volume_chart(fig, stock_data)
    #fig = _build_bband_chart(fig, stock_data)

    fig.update_yaxes(title_text='股價', row=1, col=1)
    fig.update_xaxes(rangeslider_visible=False, row=1, col=1)
    for r in range(1, 6):
        fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])], row=r, col=1)

    fig = _apply_cyber_theme(fig, stock_data, stock_number)

    text_area = _build_signal_panel(stock_data, stock_number)
    save_plt_to_html(stock_number, stock_data, fig, text_area)


In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import threading

In [17]:
global df_save_df_to_excel
global df_save_old_RunRealTimeStock
df_save_df_to_excel=pd.DataFrame()
df_save_old_RunRealTimeStock=pd.DataFrame()

def save_to_df(_df):
    _df['年月日'] = _df['年月日'].dt.strftime('%Y-%m-%d')
    
    # 處理資料 df1
    df1=_df
    
    # 處理資料 df2
    _stock_data=_df
    to_RunRealTimeStock_excel=_df
    to_RunRealTimeStock_excel['now_price']=_stock_data['收盤價']
    to_RunRealTimeStock_excel['change_price']=_stock_data['漲跌價差']
    to_RunRealTimeStock_excel['change_quote'] = (_stock_data['漲跌價差'].astype(float, errors='ignore') / _stock_data['開盤價'].astype(float, errors='ignore')).replace([float('inf'), -float('inf'), float('nan')], 0) * 100

    to_RunRealTimeStock_excel['change_quote'] = to_RunRealTimeStock_excel['change_quote'].astype(float).apply(lambda x: f"{x:.2f}%")
    df2=to_RunRealTimeStock_excel[['日期','stock_number','now_price','change_price','change_quote']]

    # 更新全局 DataFrame
    global df_save_df_to_excel  
    if  df_save_df_to_excel.empty:
        df_save_df_to_excel=df1
    else :
        df_save_df_to_excel = pd.concat([df_save_df_to_excel, df1], ignore_index=True)
       
    global df_save_old_RunRealTimeStock  
    if  df_save_old_RunRealTimeStock.empty:
        df_save_old_RunRealTimeStock=df2
    else :
        df_save_old_RunRealTimeStock = pd.concat([df_save_old_RunRealTimeStock, df2], ignore_index=True)

In [18]:
def save_process_data(stock_number, stock_data):
    try:
        if stock_data.empty:
            print(f'DataFrame is empty. Skipping save.{stock_number}')
            return
        
        stock_data['stock_number'] = stock_number
        
        # 存 excel
        _max_end_date = max(end_time_list)
        _min_end_date = min(end_time_list) 
        
        max_end_date = f"{int(_max_end_date.split('/')[0]) + 1911}-{_max_end_date.split('/')[1]}-{_max_end_date.split('/')[2]}"
        min_end_date = f"{int(_min_end_date.split('/')[0]) + 1911}-{_min_end_date.split('/')[1]}-{_min_end_date.split('/')[2]}"

        stock_data = stock_data[
            (stock_data['年月日'] <= max_end_date) & 
            (stock_data['年月日'] >= min_end_date)
        ]

        save_to_df(stock_data)
        
    except Exception as error:
        print(f'Error in save_process_data for stock {stock_number}: {error}')

# 定义主要爬虫和处理函数
def process_stock_codes(stock_number):
    try:
        #全月份重新更新
        craw_stock_need_update=True
        #global _dummyData
        
        #當下月份會重新更新 其他月份不會
        craw_stock_need_update=False
        RowData_df_craw_stock, His_Stock, isSuccess = craw_stock(stock_number, start_month,(datetime.now() - timedelta(days=0)).strftime("%Y-%m-%d"),craw_stock_need_update)
        _dummyData=data_process(RowData_df_craw_stock)

        #存html 
        gen_html(stock_number, _dummyData)  
        save_process_data( stock_number, _dummyData)

    except Exception as error:
        print(f'Error processing stock {stock_number}: {error}')

# 分別處理不同的股票列表
def process_stock_list(stock_list):
    for stock_number in stock_list:
        process_stock_codes(stock_number)

In [19]:
## 【Run】 跑數據
start_month = '2025-05-01'

start_date = datetime.strptime(start_month, "%Y-%m-%d").date()
today = datetime.today().date()

def to_roc(date_obj):
    roc_year = date_obj.year - 1911
    return f"{roc_year}/{date_obj.month:02d}/{date_obj.day:02d}"

end_time_list = []
current_date = start_date
while current_date <= today:
    end_time_list.append(to_roc(current_date))
    current_date += timedelta(days=1)

end_time_list=end_time_list[-10:]

min(end_time_list) 


'115/06/26'

# Test
df_save_df_to_excel

_dummyData

In [20]:
import time

def process_stock_codes_test(stock_number):
    try:
        craw_stock_need_update = False
        craw_stock_need_update= True
        t0 = time.time()
        RowData_df_craw_stock, His_Stock, isSuccess = craw_stock(
            stock_number, start_month,
            (datetime.now() - timedelta(days=0)).strftime("%Y-%m-%d"),
            craw_stock_need_update
        )
        t1 = time.time()
        _dummyData = data_process(RowData_df_craw_stock)
        t2 = time.time()
        gen_html(stock_number, _dummyData)
        t3 = time.time()
        save_process_data(stock_number, _dummyData)
        t4 = time.time()
        print(f'{stock_number} | craw:{t1-t0:.2f}s  process:{t2-t1:.2f}s  html:{t3-t2:.2f}s  save:{t4-t3:.2f}s')
    except Exception as error:
        print(f'Error processing stock {stock_number}: {error}')

# 覆寫

# 確認是否已被抓取過
def check_data_exist(file_type,stock_number,date):
    stock_number_=str(stock_number)
    date_=date.strftime("%Y-%m-%d")
    file_path = rf"D:\Project\Jupyter\Stock\Main\Data\{file_type}_{stock_number}_{date_}.json"
    
    return False
    
    #特定月份更新
    if('2026-06'== date.strftime("%Y-%m")):
        return False
    
    
    # alaways 跑本月 更新
    '''
    if date.strftime("%Y-%m") == datetime.now().strftime("%Y-%m"):
        print('【craw_stock】跑本當月更新 -->'+stock_number,datetime.now().strftime("%Y-%m"))
        return False
    '''
    # 1. 確認檔案是否存在
    if os.path.exists(file_path):
        # 2. 讀取檔案內容
        with open(file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
            #print(data)
        return True
    return False

process_stock_codes_test('3014')



process_stock_codes_test('1599')
process_stock_codes_test('8069')
process_stock_codes_test('1101')
process_stock_codes_test('2062')
process_stock_codes_test('6805')
process_stock_codes_test('2382')
process_stock_codes_test('3583')